# Differential ML with a Difference

Derivatives Informed Neural Network for Option Pricing

Implements differential machine learning experiments for multiple option types:
- European options (Black-Scholes)
- Basket options (Bachelier multi-dimensional)
- Digital basket options
- Barrier options (down-and-out)
- Gamma-enhanced pricing

Framework features:
- Twin-network architecture with explicit backpropagation
- Second-order derivatives for gamma
- Pathwise and Likelihood Ratio Method derivatives
- Comprehensive normalization

Reference: This work builds on the work of the working paper Differential Machine Learning by Brian Huge and Antoine Savine (2020) https://github.com/differential-machine-learning/notebooks/blob/master/DifferentialMLTF2.ipynb and follows a similar coding style and structure.

## Imports

In [ ]:
try:
  %matplotlib notebook
except Exception:
    pass

import torch
import torch.nn as nn
import torch.optim as optim

import warnings
warnings.filterwarnings('ignore')

# import other useful libs
import numpy as np
from scipy.stats import norm
import matplotlib.pyplot as plt
import time
from tqdm import tqdm_notebook
import pickle
import os

real_dtype = torch.float32

## Configuration

In [ ]:
USE_MULTIPATH = True      # Toggle: False for antithetic (2 paths), True for multipath (10 paths)
RUN_MULTIPLE_SEEDS = True # Toggle: False for single run with graphs, True for 100 seeds with statistics

NUM_SEEDS = 100             # Number of seeds when RUN_MULTIPLE_SEEDS = True
NUM_PATHS_PER_S1 = 10       # Number of paths per S1 when USE_MULTIPATH = True
GROUND_TRUTH_PATHS = 1000000
FINITE_DIFF_BUMP = 0.001

BS_SAMPLE_SIZES = [1024, 8192]
BS_NUM_TEST = 100

# digital basket configuration
BASKET_DIGITAL_CORRELATION = 0.0     # Correlation for digital basket options (can be 0.0, 0.3, 0.5, 0.8)
BASKET_DIGITAL_VOL = 0.20            # Fixed basket volatility for digital baskets
BASKET_DIGITAL_NUM_PATHS = 10        # Number of paths per sample for digital basket training

BASKET_DIGITAL_DIM1_SAMPLE_SIZES = [1024, 8192]
BASKET_DIGITAL_DIM1_NUM_TEST = 4096

BASKET_DIGITAL_DIM7_SAMPLE_SIZES = [4096, 8192, 16384]
BASKET_DIGITAL_DIM7_NUM_TEST = 4096

BASKET_DIGITAL_DIM20_SAMPLE_SIZES = [16384, 4*16384, 8*16384]
BASKET_DIGITAL_DIM20_NUM_TEST = 4096

# barrier options configuration
BARRIER_SAMPLE_SIZES = [1024, 8192]
BARRIER_NUM_TEST = 100
BARRIER_NUM_PATHS_PER_S1 = 10  # Number of paths per S1 for barrier training

# gamma options configuration
GAMMA_SAMPLE_SIZES = [1024, 8192]
GAMMA_NUM_TEST = 100
GAMMA_NUM_PATHS_PER_S1 = 10  # Number of paths per S1 for gamma training

WEIGHT_SEED = None

# barrier options configuration
BARRIER_SPOT = 1.0
BARRIER_K = 0
BARRIER_BARRIER = 0.85
BARRIER_VOL = 0.2
BARRIER_T1 = 1/3
BARRIER_T2 = 2/3
BARRIER_T3 = 1.0
BARRIER_VOL_MULT = 1.5

# gamma options configuration
GAMMA_WEIGHTS = {
    'price': 0.4,
    'delta': 0.4,
    'gamma': 0.2
}

# portfolio options configuration
PORTFOLIO_VOL = 0.20
PORTFOLIO_T1 = 1.0/6.0  # Training time (1/6 year)
PORTFOLIO_T2 = 1.0/3.0  # Maturity (1/3 year)
PORTFOLIO_VOL_MULT = 1.5
PORTFOLIO_SPOT = 1.0

# Call portfolio strikes and weights: C(0.85) - 1.5*C(0.9) + 0.75*C(1.15)
PORTFOLIO_CALL_STRIKES = [0.85, 0.90, 1.15]
PORTFOLIO_CALL_WEIGHTS = [1.0, -1.5, 0.75]

# Digital portfolio strikes and weights: D(0.75) - D(0.95) + D(1.15) - D(1.35)
PORTFOLIO_DIGITAL_STRIKES = [0.75, 0.95, 1.15, 1.35]
PORTFOLIO_DIGITAL_WEIGHTS = [1.0, -1.0, 1.0, -1.0]

PORTFOLIO_SAMPLE_SIZES = [512, 1024, 4096, 8192]
PORTFOLIO_NUM_TEST = 100
PORTFOLIO_NUM_PATHS_PER_S1 = 10
# ramp digital options configuration
RAMP_SPOT = 1.0
RAMP_STRIKE = 1.10
RAMP_VOLATILITY = 0.2
RAMP_T1 = 1.0
RAMP_T2 = 2.0
RAMP_VOL_MULT = 1.5


# Part I: Core Fucntions

## Feedforward neural network in PyTorch

This function builds a classic feedforward neural network in PyTorch, implementing the set of equations from the article.
This is classic deep learning code and a direct translation of equations in Python. We tried to keep notations consistent with the paper.

The weights are initialized with PyTorch's kaiming_uniform_ (variance scaling equivalent), since the network is initialized randomly, we also implement seed in the interest of reproducibility.

In [ ]:
class VanillaNet(nn.Module):
    """Feedforward neural network (eq. 3 from paper)."""

    def __init__(self, input_dim, hidden_units, hidden_layers, seed=None):
        super().__init__()
        if seed is not None:
            torch.manual_seed(seed)

        layers = []
        in_dim = input_dim
        for l in range(hidden_layers):
            linear = nn.Linear(in_dim, hidden_units)
            # variance_scaling equivalent: kaiming_uniform with fan_in mode
            nn.init.kaiming_uniform_(linear.weight, mode='fan_in', nonlinearity='linear')
            nn.init.zeros_(linear.bias)
            layers.append(linear)
            in_dim = hidden_units

        # output layer
        out_layer = nn.Linear(hidden_units, 1)
        nn.init.kaiming_uniform_(out_layer.weight, mode='fan_in', nonlinearity='linear')
        nn.init.zeros_(out_layer.bias)

        self.hidden_layers = nn.ModuleList(layers)
        self.out_layer = out_layer

    def forward(self, x):
        # eq.3, l=0: input layer
        z = x
        # first hidden layer: linear (no activation on pre-activation)
        z = self.hidden_layers[0](z)
        # eq. 3, l=2..L-1: softplus activation then linear
        for layer in self.hidden_layers[1:]:
            z = layer(torch.nn.functional.softplus(z))
        # eq. 3, l=L: output layer (softplus on last hidden, linear out)
        y = self.out_layer(torch.nn.functional.softplus(z))
        return y


def vanilla_net(input_dim, hidden_units, hidden_layers, seed):
    """Build and return a VanillaNet module."""
    return VanillaNet(input_dim, hidden_units, hidden_layers, seed)

## Explicit backpropagation and twin network

These functions construct the twin network by explicit backpropagation. The twin network simultaneously
and efficiently predicts values and their differentials to inputs, allowing to train on datasets augmented with differential labels
(i.e. gradients of labels to inputs).

In PyTorch we use `torch.autograd.grad` to compute the Jacobian row, mirroring the explicit backprop of the original.

In [ ]:
def compute_derivs(net, x):
    """Compute d_output/d_inputs via autograd (equivalent to explicit backprop in vanilla net)."""
    x = x.requires_grad_(True)
    y = net(x)
    # sum over batch to get scalar, then differentiate
    dy_dx = torch.autograd.grad(y.sum(), x, create_graph=True)[0]
    return y, dy_dx


def twin_net_forward(net, x):
    """Combined forward pass returning output y and differentials dy/dx."""
    return compute_derivs(net, x)

## Second-order twin network for gamma prediction

Extension for gamma-enhanced experiments using torch.autograd.grad to compute second derivatives.

In [ ]:
def second_order_derivs(net, x):
    """Second-order twin network for gamma prediction.
    Returns y, dy_dx, and diagonal of Hessian (gamma).
    """
    x = x.requires_grad_(True)
    y = net(x)
    dy_dx = torch.autograd.grad(y.sum(), x, create_graph=True)[0]

    input_dim = x.shape[1]
    if input_dim == 1:
        gamma = torch.autograd.grad(dy_dx[:, 0].sum(), x, create_graph=True)[0]
        gamma = gamma.reshape(-1, 1)
    else:
        gamma_diagonal = []
        for j in range(input_dim):
            gamma_j = torch.autograd.grad(dy_dx[:, j].sum(), x, create_graph=True, retain_graph=True)[0][:, j]
            gamma_diagonal.append(gamma_j)
        gamma = torch.stack(gamma_diagonal, dim=1)

    return y, dy_dx, gamma

## Vanilla training loop

These are classic training loops for the feedforward neural network. As customary in modern deep learning, the training set is traversed in mini-batches, where the cost function is minimized with the best practice ADAM algorithm.

In [ ]:
def vanilla_train_one_epoch(net, optimizer,
                            x_train, y_train,
                            learning_rate, batch_size):
    """Training loop for one epoch (vanilla, price only)."""
    # update learning rate
    for pg in optimizer.param_groups:
        pg['lr'] = learning_rate

    m = x_train.shape[0]
    first = 0
    last = min(batch_size, m)
    while first < m:
        xb = torch.tensor(x_train[first:last], dtype=real_dtype)
        yb = torch.tensor(y_train[first:last], dtype=real_dtype)

        optimizer.zero_grad()
        pred = net(xb)
        loss = torch.nn.functional.mse_loss(pred, yb)
        loss.backward()
        optimizer.step()

        first = last
        last = min(first + batch_size, m)

## Differential training loop

The differential training loop implements the main idea from the paper, to train twin networks on datasets augmented with differentials of labels to inputs, by minimization of a combined cost function reflecting errors in both predicted values and predicted derivatives.

In [ ]:
def diff_train_one_epoch(net, optimizer,
                         x_train, y_train, dydx_train,
                         lambda_j,
                         learning_rate, batch_size):
    """Training loop for one epoch (differential ML: price + derivatives)."""
    for pg in optimizer.param_groups:
        pg['lr'] = learning_rate

    lam = torch.tensor(lambda_j, dtype=real_dtype)  # (1, n)

    m = x_train.shape[0]
    first = 0
    last = min(batch_size, m)
    while first < m:
        xb = torch.tensor(x_train[first:last], dtype=real_dtype)
        yb = torch.tensor(y_train[first:last], dtype=real_dtype)
        db = torch.tensor(dydx_train[first:last], dtype=real_dtype)

        optimizer.zero_grad()
        pred, dpred = twin_net_forward(net, xb)

        value_loss = torch.nn.functional.mse_loss(pred, yb)
        deriv_loss = torch.nn.functional.mse_loss(dpred * lam, db * lam)
        loss = 0.5 * value_loss + 0.5 * deriv_loss
        loss.backward()
        optimizer.step()

        first = last
        last = min(first + batch_size, m)

## Gamma-enhanced training loop

Extension for gamma-enhanced experiments using second-order derivatives.

In [ ]:
def gamma_train_one_epoch(net, optimizer,
                          x_train, y_train, dydx_train, d2ydx2_train,
                          w_price, w_delta, w_gamma,
                          lambda_delta, lambda_gamma,
                          learning_rate, batch_size):
    """Training loop for one epoch (gamma-enhanced: price + delta + gamma)."""
    for pg in optimizer.param_groups:
        pg['lr'] = learning_rate

    lam_d = torch.tensor(lambda_delta, dtype=real_dtype)
    lam_g = torch.tensor(lambda_gamma, dtype=real_dtype)

    m = x_train.shape[0]
    first = 0
    last = min(batch_size, m)
    while first < m:
        xb = torch.tensor(x_train[first:last], dtype=real_dtype)
        yb = torch.tensor(y_train[first:last], dtype=real_dtype)
        db = torch.tensor(dydx_train[first:last], dtype=real_dtype)
        gb = torch.tensor(d2ydx2_train[first:last], dtype=real_dtype)

        optimizer.zero_grad()
        pred, dpred, gpred = second_order_derivs(net, xb)

        loss_price = torch.nn.functional.mse_loss(pred, yb)
        loss_delta = torch.nn.functional.mse_loss(dpred * lam_d, db * lam_d)
        loss_gamma = torch.nn.functional.mse_loss(gpred * lam_g, gb * lam_g)
        loss = w_price * loss_price + w_delta * loss_delta + w_gamma * loss_gamma
        loss.backward()
        optimizer.step()

        first = last
        last = min(first + batch_size, m)

## Combined outer training loop

The outer training loop optimizes the weights of neural approximators for a number of epochs (complete sweeps of the training set).
100 epochs is more than sufficient in most practical cases. A convergence and/or cross-validation test
may be included for early stopping. Typical training takes around a second on a decent GPU (longer on Colab's shared GPUs). The
approximator class, defined next, holds all the necessary data and parameters, along with the TensorFlow graph and computing session.

In [ ]:
def train(description,
          # neural approximator
          approximator,
          # training params
          reinit=True,
          epochs=100,
          # one-cycle learning rate schedule
          learning_rate_schedule=[    (0.0, 1.0e-8),
                                      (0.2, 0.1),
                                      (0.6, 0.01),
                                      (0.9, 1.0e-6),
                                      (1.0, 1.0e-8)  ],
          batches_per_epoch=16,
          min_batch_size=256,
          # callback function and when to call it
          callback=None,           # arbitrary callable
          callback_epochs=[]):     # call after what epochs, e.g. [5, 20]

    # batching
    batch_size = max(min_batch_size, approximator.m // batches_per_epoch)

    # one-cycle learning rate schedule
    lr_schedule_epochs, lr_schedule_rates = zip(*learning_rate_schedule)

    # reset (reinitialize weights)
    if reinit:
        approximator._init_net()

    # callback on epoch 0, if requested
    if callback and 0 in callback_epochs:
        callback(approximator, 0)

    # loop on epochs, with progress bar (tqdm)
    for epoch in tqdm_notebook(range(epochs), desc=description):

        # interpolate learning rate in cycle
        learning_rate = np.interp(epoch / epochs, lr_schedule_epochs, lr_schedule_rates)

        # train one epoch
        if not approximator.differential:

            vanilla_train_one_epoch(
                approximator.net,
                approximator.optimizer,
                approximator.x,
                approximator.y,
                learning_rate,
                batch_size)

        else:

            if approximator.use_gamma:
                # gamma-enhanced training
                gamma_train_one_epoch(
                    approximator.net,
                    approximator.optimizer,
                    approximator.x,
                    approximator.y,
                    approximator.dy_dx,
                    approximator.d2y_dx2,
                    GAMMA_WEIGHTS['price'],
                    GAMMA_WEIGHTS['delta'],
                    GAMMA_WEIGHTS['gamma'],
                    approximator.lambda_delta,
                    approximator.lambda_gamma,
                    learning_rate,
                    batch_size)
            else:
                # standard differential training
                diff_train_one_epoch(
                    approximator.net,
                    approximator.optimizer,
                    approximator.x,
                    approximator.y,
                    approximator.dy_dx,
                    approximator.lambda_j,
                    learning_rate,
                    batch_size)

        # callback, if requested
        if callback and epoch in callback_epochs:
            callback(approximator, epoch)

    # final callback, if requested
    if callback and epochs in callback_epochs:
        callback(approximator, epochs)

## Data normalization

The practical performance of neural networks strongly depends on implementation details, like weight initialization and optimization.
Another crucial practicality is the normalization of training data. We refer to deep learning textbooks for a discussion of the
importance of normalization. One reason is that we need hyperparameters like the learning rate schedule to remain constant over
datasets. If notional was to be increased by factor 1M all things equal, gradients would be multiplied by 1M too and learning
rates would have to be divided by 1M to keep things similar. Normalizing data avoids manual tinkering of hyperparameters for
different datasets.

We implement below a basic normalization strategy, where the training inputs and labels are normalized by mean and standard
deviation, with differentials normalized accordingly. The differential weights in the cost function λ_j divide costs by the
norm of the normalized differentials, keeping similar the magnitude of all the components of the cost.

In [ ]:
# basic data preparation
epsilon = 1.0e-08
def normalize_data(x_raw, y_raw, dydx_raw=None, crop=None):

    # crop dataset
    m = crop if crop is not None else x_raw.shape[0]
    x_cropped = x_raw[:m]
    y_cropped = y_raw[:m]
    dycropped_dxcropped = dydx_raw[:m] if dydx_raw is not None else None

    # normalize dataset
    x_mean = x_cropped.mean(axis=0)
    x_std = x_cropped.std(axis=0) + epsilon
    x = (x_cropped- x_mean) / x_std
    y_mean = y_cropped.mean(axis=0)
    y_std = y_cropped.std(axis=0) + epsilon
    y = (y_cropped-y_mean) / y_std

    # normalize derivatives too
    if dycropped_dxcropped is not None:
        dy_dx = dycropped_dxcropped / y_std * x_std
        # weights of derivatives in cost function = (quad) mean size
        # The formula lambda_j = 1.0 / sqrt(variance) balances loss components
        dy_dx_var = (dy_dx ** 2).mean(axis=0)

        # Handle zero variance: dimensions with zero variance provide no useful information,
        # so we give them moderate weight (1.0) instead of infinite weight.
        # For non-zero variance dimensions, use the standard formula: 1.0 / sqrt(variance)
        # This correctly handles cases where all, some, or no dimensions have zero variance.
        lambda_j = np.where(dy_dx_var < 1e-12, 1.0, 1.0 / np.sqrt(dy_dx_var)).reshape(1, -1)
    else:
        dy_dx = None
        lambda_j = None

    return x_mean, x_std, x, y_mean, y_std, y, dy_dx, lambda_j

# extended data preparation with gamma
def normalize_data_with_gamma(x_raw, y_raw, dydx_raw=None, d2ydx2_raw=None, crop=None):

    # crop dataset
    m = crop if crop is not None else x_raw.shape[0]
    x_cropped = x_raw[:m]
    y_cropped = y_raw[:m]
    d1_cropped = dydx_raw[:m] if dydx_raw is not None else None
    d2_cropped = d2ydx2_raw[:m] if d2ydx2_raw is not None else None

    # normalize dataset
    x_mean = x_cropped.mean(axis=0)
    x_std = x_cropped.std(axis=0) + epsilon
    x = (x_cropped- x_mean) / x_std
    y_mean = y_cropped.mean(axis=0)
    y_std = y_cropped.std(axis=0) + epsilon
    y = (y_cropped-y_mean) / y_std

    # normalize first derivatives
    if d1_cropped is not None:
        dy_dx = d1_cropped / y_std * x_std
        # weights of first derivatives in cost function
        dy_dx_var = (dy_dx ** 2).mean(axis=0)
        # Handle zero variance: dimensions with zero variance get moderate weight (1.0)
        lambda_delta = np.where(dy_dx_var < 1e-12, 1.0, 1.0 / np.sqrt(dy_dx_var)).reshape(1, -1)
    else:
        dy_dx = None
        lambda_delta = None

    # normalize second derivatives
    if d2_cropped is not None:
        d2y_dx2 = d2_cropped / y_std * (x_std ** 2)
        # weights of second derivatives in cost function
        d2y_dx2_var = (d2y_dx2 ** 2).mean(axis=0)
        # Handle zero variance: dimensions with zero variance get moderate weight (1.0)
        lambda_gamma = np.where(d2y_dx2_var < 1e-12, 1.0, 1.0 / np.sqrt(d2y_dx2_var)).reshape(1, -1)
    else:
        d2y_dx2 = None
        lambda_gamma = None

    return x_mean, x_std, x, y_mean, y_std, y, dy_dx, d2y_dx2, lambda_delta, lambda_gamma

## Putting it all together

For convenience, we put it all together in a *Neural_Approximator* class. Most of the code should be self explanatory.

Note that we compute the coefficients α and β for balancing cost between values and derivatives in a straightforward manner:

    α = 1 / (1 + λ n)  and  β = λ n / (1 + λ n)

where n is the number of inputs, so an error on a derivative has a weight similar to a value error, and λ is a hyperparameter without significant effect, as explained in the paper, and left to 1, safe for debugging.

We implement a simple, feedforward architecture with 4 hidden layers of 20 units, throughout our tests.

Extensions:
- Gamma-enhanced training with second-order derivatives
- LRM method support for digital options (toggle between pathwise and LRM derivatives)

In [ ]:
class Neural_Approximator():

    def __init__(self, x_raw, y_raw,
                 dydx_raw=None,      # derivatives labels
                 d2ydx2_raw=None):   # second derivatives labels (gamma)

        self.x_raw = x_raw
        self.y_raw = y_raw
        self.dydx_raw = dydx_raw
        self.d2ydx2_raw = d2ydx2_raw

        self.net = None
        self.optimizer = None

    def _init_net(self):
        """(Re)initialize network weights."""
        if self.net is not None:
            # re-initialize in-place by creating a fresh net with the same seed
            new_net = VanillaNet(self.n, self.hidden_units, self.hidden_layers, self.weight_seed)
            self.net.load_state_dict(new_net.state_dict())
        # optimizer state reset
        self.optimizer = optim.Adam(self.net.parameters())

    def build_graph(self,
                differential,       # differential or not
                use_gamma,          # use gamma (second derivatives)
                lam,                # balance cost between values and derivs
                hidden_units,
                hidden_layers,
                weight_seed):

        self.differential = differential
        self.use_gamma = use_gamma
        self.hidden_units = hidden_units
        self.hidden_layers = hidden_layers
        self.weight_seed = weight_seed

        # Build the PyTorch network
        self.net = VanillaNet(self.n, hidden_units, hidden_layers, weight_seed)
        self.optimizer = optim.Adam(self.net.parameters())

    # prepare for training with m examples, standard or differential
    def prepare(self,
                m,
                differential,
                use_gamma=False,     # use gamma (second derivatives)
                lam=1,               # balance cost between values and derivs
                # standard architecture
                hidden_units=20,
                hidden_layers=4,
                weight_seed=None):

        # prepare dataset
        if use_gamma:
            # gamma-enhanced normalization
            self.x_mean, self.x_std, self.x, self.y_mean, self.y_std, self.y, \
            self.dy_dx, self.d2y_dx2, self.lambda_delta, self.lambda_gamma = \
                normalize_data_with_gamma(self.x_raw, self.y_raw, self.dydx_raw, self.d2ydx2_raw, m)
        else:
            # standard normalization
            self.x_mean, self.x_std, self.x, self.y_mean, self.y_std, self.y, self.dy_dx, self.lambda_j = \
                normalize_data(self.x_raw, self.y_raw, self.dydx_raw, m)

        # build graph
        self.m, self.n = self.x.shape
        self.build_graph(differential, use_gamma, lam, hidden_units, hidden_layers, weight_seed)

    def train(self,
              description="training",
              # training params
              reinit=True,
              epochs=100,
              # one-cycle learning rate schedule
              learning_rate_schedule=[
                  (0.0, 1.0e-8),
                  (0.2, 0.1),
                  (0.6, 0.01),
                  (0.9, 1.0e-6),
                  (1.0, 1.0e-8)],
              batches_per_epoch=16,
              min_batch_size=256,
              # callback and when to call it
              callback=None,           # arbitrary callable
              callback_epochs=[]):     # call after what epochs, e.g. [5, 20]

        train(description,
              self,
              reinit,
              epochs,
              learning_rate_schedule,
              batches_per_epoch,
              min_batch_size,
              callback,
              callback_epochs)

    def predict_values(self, x):
        # scale
        x_scaled = (x - self.x_mean) / self.x_std
        x_t = torch.tensor(x_scaled, dtype=real_dtype)
        self.net.eval()
        with torch.no_grad():
            y_scaled = self.net(x_t).numpy()
        self.net.train()
        # unscale
        y = self.y_mean + self.y_std * y_scaled
        return y

    def predict_values_and_derivs(self, x):
        # scale
        x_scaled = (x - self.x_mean) / self.x_std
        x_t = torch.tensor(x_scaled, dtype=real_dtype).requires_grad_(True)
        self.net.eval()
        y_scaled_t = self.net(x_t)
        dy_scaled = torch.autograd.grad(y_scaled_t.sum(), x_t)[0]
        self.net.train()
        # unscale
        y = self.y_mean + self.y_std * y_scaled_t.detach().numpy()
        dydx = self.y_std / self.x_std * dy_scaled.detach().numpy()
        return y, dydx

    def predict_values_derivs_and_gamma(self, x):
        # scale
        x_scaled = (x - self.x_mean) / self.x_std
        x_t = torch.tensor(x_scaled, dtype=real_dtype).requires_grad_(True)
        self.net.eval()
        y_scaled_t = self.net(x_t)
        dy_scaled = torch.autograd.grad(y_scaled_t.sum(), x_t, create_graph=True)[0]

        input_dim = x_t.shape[1]
        if input_dim == 1:
            gamma_scaled = torch.autograd.grad(dy_scaled[:, 0].sum(), x_t)[0]
            gamma_scaled = gamma_scaled.reshape(-1, 1)
        else:
            gamma_diagonal = []
            for j in range(input_dim):
                gj = torch.autograd.grad(dy_scaled[:, j].sum(), x_t, retain_graph=True)[0][:, j]
                gamma_diagonal.append(gj)
            gamma_scaled = torch.stack(gamma_diagonal, dim=1)

        self.net.train()
        y = self.y_mean + self.y_std * y_scaled_t.detach().numpy()
        dydx = self.y_std / self.x_std * dy_scaled.detach().numpy()
        gamma = self.y_std / (self.x_std ** 2) * gamma_scaled.detach().numpy()
        return y, dydx, gamma

# Part II : Dealing with Discontinuities

## Digital Options: The Challenge of Discontinuous Payoffs

Digital options present a unique challenge for differential machine learning. Unlike standard options where the payoff
is smooth and continuous (max(S-K, 0)), digital options have a **discontinuous payoff** (1 if S > K, else 0). This
discontinuity creates problems for pathwise derivative estimation.

### The Problem with Pathwise Derivatives

For a digital option, the pathwise derivative involves the Dirac delta function:

    ∂payoff/∂S1 = δ(S2 - K) * ∂S2/∂S1

where δ is the Dirac delta. In practice, this must be approximated. The standard approach uses the analytical delta:

    digital_delta = N'(d2) / (S1 * σ * √T)

where N' is the standard normal PDF and d2 is from Black-Scholes. While mathematically correct, this approximation
can be unstable for differential ML training, especially with limited samples.

### The LRM (Likelihood Ratio Method) Alternative

The LRM method provides an elegant solution by using the **score function** instead of pathwise differentiation:

    ∂E[payoff]/∂S1 = E[payoff * score]

For digital options in the Black-Scholes model, this becomes:

    LRM_derivative = indicator(S2 >= K) * (Z / (S1 * σ * √T))

where Z is the standard normal random variable used to simulate S2. The key advantage is that the indicator function
(0 or 1) is used directly without differentiation, avoiding the delta function entirely.

### Why LRM Works Better for Digital Options

1. **No discontinuity issues**: LRM uses the indicator directly, no delta function approximation needed
2. **Unbiased estimator**: Provides an unbiased estimate of the true derivative
3. **Better sample efficiency**: More stable with small training sets
4. **Natural extension**: Works seamlessly for path-dependent options (barriers) using product of scores

### The Experiment

We compare three methods for digital option pricing:

1. **Standard ML**: Vanilla neural network, no derivatives used
2. **Differential ML (Pathwise)**: Twin network with analytical digital deltas
3. **Differential ML (LRM)**: Twin network with LRM-based derivatives

The results demonstrate that LRM-based differential ML achieves superior accuracy, particularly for delta predictions,
validating its effectiveness for discontinuous payoffs.

In [ ]:
# helper analytics for standard European options
def bsPrice(spot, strike, vol, T):
    """Black-Scholes price for European call"""
    d1 = (np.log(spot/strike) + 0.5 * vol * vol * T) / vol / np.sqrt(T)
    d2 = d1 - vol * np.sqrt(T)
    return spot * norm.cdf(d1) - strike * norm.cdf(d2)

def bsDelta(spot, strike, vol, T):
    """Black-Scholes delta for European call"""
    d1 = (np.log(spot/strike) + 0.5 * vol * vol * T) / vol / np.sqrt(T)
    return norm.cdf(d1)

def bsVega(spot, strike, vol, T):
    """Black-Scholes vega for European call"""
    d1 = (np.log(spot/strike) + 0.5 * vol * vol * T) / vol / np.sqrt(T)
    return spot * np.sqrt(T) * norm.pdf(d1)

def bsGamma(spot, strike, vol, T):
    """Black-Scholes gamma for European call"""
    d1 = (np.log(spot/strike) + 0.5 * vol * vol * T) / vol / np.sqrt(T)
    return norm.pdf(d1) / (spot * vol * np.sqrt(T))

# helper analytics for digital options
def digitalPrice(spot, strike, vol, T):
    """Analytical price for digital call option"""
    if T <= 0:
        return (spot > strike).astype(float)
    d2 = (np.log(spot/strike) / (vol * np.sqrt(T))) - (vol * np.sqrt(T) / 2)
    return norm.cdf(d2)

def digitalDelta(spot, strike, vol, T):
    """Analytical delta for digital call option"""
    if T <= 0:
        return 0.0
    d2 = (np.log(spot/strike) / (vol * np.sqrt(T))) - (vol * np.sqrt(T) / 2)
    return norm.pdf(d2) / (spot * vol * np.sqrt(T))

def digitalGamma(spot, strike, vol, T):
    """Analytical gamma for digital call option (second derivative of price)"""
    if T <= 0:
        return 0.0
    d2 = (np.log(spot/strike) / (vol * np.sqrt(T))) - (vol * np.sqrt(T) / 2)
    # Gamma = -d2 * norm.pdf(d2) / (spot * vol * sqrt(T))
    return -d2 * norm.pdf(d2) / (spot * vol * np.sqrt(T))

## LRM (Likelihood Ratio Method) Derivative Calculation

In [ ]:
def lrm_derivative(S1, S2, K, vol, T, Z):
    """
    Calculate LRM-based derivative for digital options using the score function

    Args:
        S1: Spot price at time T1 (array)
        S2: Spot price at time T2 - simulated (array)
        K: Strike price (scalar)
        vol: Volatility (scalar)
        T: Time to maturity T2-T1 (scalar)
        Z: Standard normal random variable used in simulation (array)

    Returns:
        LRM derivative: indicator * (Z / (S1 * vol * sqrt(T)))

    The score function approach:
    - indicator = 1 if S2 >= K, 0 otherwise (the digital payoff)
    - score = Z / (S1 * vol * sqrt(T)) (derivative of log-likelihood)
    - LRM derivative = payoff * score (unbiased estimator)

    This avoids the discontinuity in pathwise derivatives by using the score of the
    distribution rather than differentiating the payoff directly.
    """
    indicator = np.where(S2 >= K, 1.0, 0.0)
    lrm_deriv = indicator * (Z / (S1 * vol * np.sqrt(T)))
    return lrm_deriv

## Black-Scholes model with digital option support

This extends the standard Black-Scholes simulator to support both standard European calls and digital options.
For digital options, we generate training sets with **both** pathwise and LRM derivatives, allowing direct
comparison of the two methods.

Key features:
- Standard training set: European calls with pathwise derivatives (as in Part I)
- Digital training set: Binary payoffs with both pathwise AND LRM derivatives
- Antithetic variance reduction for both option types
- Test sets with analytical ground truth prices and deltas

In [ ]:
class BlackScholes:

    def __init__(self,
                 vol=0.2,
                 T1=1,
                 T2=2,
                 K=1.10,
                 volMult=1.5):

        self.spot = 1
        self.vol = vol
        self.T1 = T1
        self.T2 = T2
        self.K = K
        self.volMult = volMult

    # training set for standard European calls: returns S1 (mx1), payoff (mx1) and pathwise delta (mx1)
    def trainingSet(self, m, anti=True, seed=None):

        np.random.seed(seed)

        # 2 sets of normal returns
        returns = np.random.normal(size=[m, 2])

        # SDE
        vol0 = self.vol * self.volMult
        R1 = np.exp(-0.5*vol0*vol0*self.T1 + vol0*np.sqrt(self.T1)*returns[:,0])
        R2 = np.exp(-0.5*self.vol*self.vol*(self.T2-self.T1) \
                    + self.vol*np.sqrt(self.T2-self.T1)*returns[:,1])
        S1 = self.spot * R1
        S2 = S1 * R2

        # payoff
        pay = np.maximum(0, S2 - self.K)

        # two antithetic paths
        if anti:

            R2a = np.exp(-0.5*self.vol*self.vol*(self.T2-self.T1) \
                    - self.vol*np.sqrt(self.T2-self.T1)*returns[:,1])
            S2a = S1 * R2a
            paya = np.maximum(0, S2a - self.K)

            X = S1
            Y = 0.5 * (pay + paya)

            # pathwise differentials
            Z1 =  np.where(S2 > self.K, R2, 0.0).reshape((-1,1))
            Z2 =  np.where(S2a > self.K, R2a, 0.0).reshape((-1,1))
            Z = 0.5 * (Z1 + Z2)

        # standard
        else:

            X = S1
            Y = pay

            # pathwise differentials
            Z =  np.where(S2 > self.K, R2, 0.0).reshape((-1,1))

        return X.reshape([-1,1]), Y.reshape([-1,1]), Z.reshape([-1,1])

    # training set for digital options: returns S1, binary payoff, pathwise delta, AND lrm delta
    def digitalTrainingSet(self, m, anti=True, seed=None):
        """
        Generate training set for digital options with BOTH pathwise and LRM derivatives

        Returns:
            X: S1 values (mx1)
            Y: Digital payoffs (mx1) - probability from analytical formula
            Z_pathwise: Pathwise deltas (mx1) - explicitly set to 0 for digital options
            Z_lrm: LRM-based derivatives (mx1)

        For comparison experiments, we return both derivative types so we can train
        separate models with pathwise vs LRM and compare their performance.
        """
        np.random.seed(seed)

        # 2 sets of normal returns
        returns = np.random.normal(size=[m, 2])

        # SDE - same as standard option
        vol0 = self.vol * self.volMult
        R1 = np.exp(-0.5*vol0*vol0*self.T1 + vol0*np.sqrt(self.T1)*returns[:,0])
        R2 = np.exp(-0.5*self.vol*self.vol*(self.T2-self.T1) \
                    + self.vol*np.sqrt(self.T2-self.T1)*returns[:,1])
        S1 = self.spot * R1
        S2 = S1 * R2

        # digital payoff: binary indicator (consistent with multipath version)
        pay = np.where(S2 >= self.K, 1.0, 0.0)

        # two antithetic paths
        if anti:

            R2a = np.exp(-0.5*self.vol*self.vol*(self.T2-self.T1) \
                    - self.vol*np.sqrt(self.T2-self.T1)*returns[:,1])
            S2a = S1 * R2a
            paya = np.where(S2a >= self.K, 1.0, 0.0)

            X = S1
            Y = 0.5 * (pay + paya)  # Average binary payoffs

            # pathwise derivatives: explicitly set to 0 for digital options (discontinuous payoff)
            Z_pathwise = np.zeros_like(S1)

            # LRM derivatives: indicator * score
            Z_values1 = returns[:, 1]
            Z_values2 = -returns[:, 1]  # antithetic
            indicator1 = np.where(S2 >= self.K, 1.0, 0.0)
            indicator2 = np.where(S2a >= self.K, 1.0, 0.0)
            lrm1 = indicator1 * (Z_values1 / (S1 * self.vol * np.sqrt(self.T2-self.T1)))
            lrm2 = indicator2 * (Z_values2 / (S1 * self.vol * np.sqrt(self.T2-self.T1)))
            Z_lrm = 0.5 * (lrm1 + lrm2)

        # standard (no antithetic)
        else:

            X = S1
            Y = pay

            # pathwise derivative: set to 0 for digital options (discontinuous payoff)
            Z_pathwise = np.zeros_like(S1)

            # LRM derivative
            Z_values = returns[:, 1]
            indicator = np.where(S2 >= self.K, 1.0, 0.0)
            Z_lrm = indicator * (Z_values / (S1 * self.vol * np.sqrt(self.T2-self.T1)))

        return X.reshape([-1,1]), Y.reshape([-1,1]), Z_pathwise.reshape([-1,1]), Z_lrm.reshape([-1,1])

    # multipath training set for digital options: averages over multiple paths per S1
    def digitalMultiPathTrainingSet(self, m, num_paths=10, seed=None):
        """
        Generate training set for digital options with multiple paths per S1

        Instead of 1 or 2 paths (antithetic), we generate num_paths S2 values for each S1.
        This produces smoother labels by averaging multiple binary outcomes.

        Args:
            m: Number of S1 samples
            num_paths: Number of S2 paths per S1 (default 10)
            seed: Random seed

        Returns:
            X: S1 values (mx1)
            Y: Average digital payoffs over num_paths (mx1)
            Z_pathwise: Average pathwise derivatives (mx1)
            Z_lrm: Average LRM derivatives (mx1)
        """
        np.random.seed(seed)

        # generate S1 values
        returns1 = np.random.normal(size=m)
        vol0 = self.vol * self.volMult
        R1 = np.exp(-0.5*vol0*vol0*self.T1 + vol0*np.sqrt(self.T1)*returns1)
        S1 = self.spot * R1

        # generate num_paths S2 values for each S1
        returns2 = np.random.normal(size=[m, num_paths])

        # simulate all paths at once
        R2 = np.exp(-0.5*self.vol*self.vol*(self.T2-self.T1) \
                    + self.vol*np.sqrt(self.T2-self.T1)*returns2)
        S2 = S1.reshape(-1, 1) * R2  # (m x num_paths)

        # Use binary payoffs averaged across paths (gives non-binary payoff)
        binary_payoffs = np.where(S2 >= self.K, 1.0, 0.0)  # Binary: 1 if S2 >= K, 0 otherwise

        # pathwise derivatives: explicitly set to 0 for digital options (discontinuous payoff)
        pathwise_derivs = np.zeros_like(S2)

        # LRM derivatives: indicator * (Z / (S1 * vol * sqrt(T)))
        indicator = np.where(S2 >= self.K, 1.0, 0.0)
        lrm_derivs = indicator * (returns2 / (S1.reshape(-1, 1) * self.vol * np.sqrt(self.T2-self.T1)))

        # Average across paths
        X = S1
        Y = binary_payoffs.mean(axis=1)  # average binary payoffs (gives non-binary result)
        Z_pathwise = pathwise_derivs.mean(axis=1)  # average pathwise derivatives
        Z_lrm = lrm_derivs.mean(axis=1)  # average LRM derivatives

        return X.reshape([-1,1]), Y.reshape([-1,1]), Z_pathwise.reshape([-1,1]), Z_lrm.reshape([-1,1])

    # test set for standard options: returns grid of spots with analytical prices, deltas and vegas
    def testSet(self, lower=0.35, upper=1.65, num=100, seed=None):

        spots = np.linspace(lower, upper, num).reshape((-1, 1))
        # compute prices, deltas and vegas using Black-Scholes formulas
        prices = bsPrice(spots, self.K, self.vol, self.T2 - self.T1).reshape((-1, 1))
        deltas = bsDelta(spots, self.K, self.vol, self.T2 - self.T1).reshape((-1, 1))
        vegas = bsVega(spots, self.K, self.vol, self.T2 - self.T1).reshape((-1, 1))
        gammas = bsGamma(spots, self.K, self.vol, self.T2 - self.T1).reshape((-1, 1))
        return spots, spots, prices, deltas, gammas

    # test set for digital options: returns grid of spots with analytical digital prices and deltas
    def digitalTestSet(self, lower=0.35, upper=1.65, num=100, seed=None):
        """
        Generate test data for digital options using analytical formulas

        Returns ground truth for evaluating both pathwise and LRM-trained models
        """
        spots = np.linspace(lower, upper, num).reshape((-1, 1))
        # compute digital prices and deltas analytically
        prices = digitalPrice(spots, self.K, self.vol, self.T2 - self.T1).reshape((-1, 1))
        deltas = digitalDelta(spots, self.K, self.vol, self.T2 - self.T1).reshape((-1, 1))
        # no vega for now
        vegas = np.zeros_like(prices)
        return spots, spots, prices, deltas, vegas

## Digital options comparison experiment

The test function below trains **three** types of neural approximators for digital options:
1. Standard ML (vanilla network, no derivatives)
2. Differential ML with pathwise derivatives (analytical digital deltas)
3. Differential ML with LRM derivatives (score function)

All three models train on the same simulated prices, but with different derivative information.
We then compare their performance on analytical ground truth.

In [ ]:
def test_digital(generator,
                 sizes,
                 nTest,
                 simulSeed=None,
                 testSeed=None,
                 weightSeed=None,
                 deltidx=0):

    # simulation - choose between antithetic or multipath
    if USE_MULTIPATH:
        xTrain, yTrain, dydxTrain_pathwise, dydxTrain_lrm = \
            generator.digitalMultiPathTrainingSet(max(sizes), num_paths=NUM_PATHS_PER_S1, seed=simulSeed)
    else:
        xTrain, yTrain, dydxTrain_pathwise, dydxTrain_lrm = \
            generator.digitalTrainingSet(max(sizes), seed=simulSeed)

    xTest, xAxis, yTest, dydxTest, vegas = \
        generator.digitalTestSet(num=nTest, seed=testSeed)

    # neural approximators - one for each method
    # all three use the same prices (yTrain) but different derivative information
    regressor_standard = Neural_Approximator(xTrain, yTrain, None)
    regressor_pathwise = Neural_Approximator(xTrain, yTrain, dydxTrain_pathwise)
    regressor_lrm = Neural_Approximator(xTrain, yTrain, dydxTrain_lrm)

    predvalues = {}
    preddeltas = {}

    for size in sizes:

        # 1. Standard ML (no derivatives)
        regressor_standard.prepare(size, False, weight_seed=weightSeed)
        regressor_standard.train("standard training")
        predictions, deltas = regressor_standard.predict_values_and_derivs(xTest)
        predvalues[("standard", size)] = predictions
        preddeltas[("standard", size)] = deltas[:, deltidx]

        # 2. Differential ML with pathwise derivatives
        regressor_pathwise.prepare(size, True, weight_seed=weightSeed)
        regressor_pathwise.train("differential training")
        predictions, deltas = regressor_pathwise.predict_values_and_derivs(xTest)
        predvalues[("differential", size)] = predictions
        preddeltas[("differential", size)] = deltas[:, deltidx]

        # 3. Differential ML with LRM derivatives
        regressor_lrm.prepare(size, True, weight_seed=weightSeed)
        regressor_lrm.train("lrm training")
        predictions, deltas = regressor_lrm.predict_values_and_derivs(xTest)
        predvalues[("lrm", size)] = predictions
        preddeltas[("lrm", size)] = deltas[:, deltidx]

    return xAxis, yTest, dydxTest[:, deltidx], vegas, predvalues, preddeltas

def graph_digital(title,
                  predictions,
                  xAxis,
                  xAxisName,
                  yAxisName,
                  targets,
                  sizes,
                  computeRmse=False,
                  weights=None):
    """
    Graph for digital options showing 3 methods: standard, differential (pathwise), lrm
    """
    numRows = len(sizes)
    numCols = 3  # standard, differential, lrm

    fig, ax = plt.subplots(numRows, numCols, squeeze=False)
    fig.set_size_inches(4 * numCols + 1.5, 4 * numRows)

    # row labels (sizes)
    for i, size in enumerate(sizes):
        ax[i,0].annotate("size %d" % size, xy=(0, 0.5),
          xytext=(-ax[i,0].yaxis.labelpad-5, 0),
          xycoords=ax[i,0].yaxis.label, textcoords='offset points',
          ha='right', va='center')

    # column titles
    ax[0,0].set_title("standard")
    ax[0,1].set_title("differential")
    ax[0,2].set_title("lrm")

    # plot each method
    for i, size in enumerate(sizes):
        for j, regType, in enumerate(["standard", "differential", "lrm"]):

            if computeRmse:
                errors = 100 * (predictions[(regType, size)] - targets)
                if weights is not None:
                    errors /= weights
                rmse = np.sqrt((errors ** 2).mean(axis=0))
                t = "rmse %.2f" % rmse
            else:
                t = xAxisName

            ax[i,j].set_xlabel(t)
            ax[i,j].set_ylabel(yAxisName)

            ax[i,j].plot(xAxis*100, predictions[(regType, size)]*100, 'co', \
                         markersize=2, markerfacecolor='white', label="predicted")
            ax[i,j].plot(xAxis*100, targets*100, 'o', color='darkred', markersize=1.5, label='targets')

            ax[i,j].legend(prop={'size': 8}, loc='upper left')

    plt.tight_layout()
    plt.subplots_adjust(top=0.9)
    plt.suptitle("% s -- %s" % (title, yAxisName), fontsize=16)
    plt.show()

## Multiple seeds experiment runner

For statistical robustness, we can run the same experiment multiple times with different random seeds
and aggregate the results. This provides mean, std, min, max statistics for RMSE.

In [ ]:
def run_digital_experiment_with_seeds(generator, sizes, nTest, num_seeds=100, weightSeed=None):
    """
    Run digital options experiment multiple times with different seeds

    Returns averaged predictions and statistics across all seeds
    """

    # storage for predictions and RMSE values across seeds
    all_predictions_values = {method: {size: [] for size in sizes} for method in ["standard", "differential", "lrm"]}
    all_predictions_deltas = {method: {size: [] for size in sizes} for method in ["standard", "differential", "lrm"]}
    rmse_values = {method: {size: [] for size in sizes} for method in ["standard", "differential", "lrm"]}
    rmse_deltas = {method: {size: [] for size in sizes} for method in ["standard", "differential", "lrm"]}

    # run experiment for each seed with progress bar
    for seed_idx in tqdm_notebook(range(num_seeds), desc="running seeds"):
        simulSeed = np.random.randint(0, 10000)

        # run experiment - FIXED: use fixed testSeed=42 so test set is same across all seeds
        xAxis, yTest, dydxTest, vegas, values, deltas = \
            test_digital(generator, sizes, nTest, simulSeed, 42, weightSeed)

        # store predictions and compute RMSE for each method and size
        for method in ["standard", "differential", "lrm"]:
            for size in sizes:
                # store predictions
                all_predictions_values[method][size].append(values[(method, size)])
                all_predictions_deltas[method][size].append(deltas[(method, size)])

                # price RMSE
                rmse_val = np.sqrt(((values[(method, size)] - yTest)**2).mean())
                rmse_values[method][size].append(rmse_val)

                # delta RMSE
                rmse_delt = np.sqrt(((deltas[(method, size)] - dydxTest)**2).mean())
                rmse_deltas[method][size].append(rmse_delt)

    # compute averaged predictions
    avg_values = {}
    avg_deltas = {}
    for method in ["standard", "differential", "lrm"]:
        for size in sizes:
            avg_values[(method, size)] = np.mean(all_predictions_values[method][size], axis=0)
            avg_deltas[(method, size)] = np.mean(all_predictions_deltas[method][size], axis=0)

    # compute statistics
    stats = {}
    for metric, data in [("price", rmse_values), ("delta", rmse_deltas)]:
        stats[metric] = {}
        for method in ["standard", "differential", "lrm"]:
            stats[metric][method] = {}
            for size in sizes:
                values_array = np.array(data[method][size])
                stats[metric][method][size] = {
                    'mean': values_array.mean(),
                    'std': values_array.std(),
                    'min': values_array.min(),
                    'max': values_array.max()
                }

    return xAxis, yTest, dydxTest, stats, avg_values, avg_deltas

def print_statistics(stats, sizes):
    """Print formatted statistics for multiple seed runs"""

    for metric in ["price", "delta"]:
        print(f"\n{metric.upper()} RMSE (over {NUM_SEEDS} seeds):")
        print(f"{'Method':<15} {'Size':<10} {'Mean':<12} {'Std':<12} {'Min':<12} {'Max':<12}")
        print("-"*70)

        for method in ["standard", "differential", "lrm"]:
            for size in sizes:
                s = stats[metric][method][size]
                print(f"{method:<15} {size:<10} {s['mean']:<12.6f} {s['std']:<12.6f} {s['min']:<12.6f} {s['max']:<12.6f}")

In [ ]:
%matplotlib inline

# simulation set sizes to perform
sizes = [1024, 8192]

# show delta?
showDeltas = True

# seed
weightSeed = None

# number of test scenarios
nTest = 100

# go
generator = BlackScholes()

if RUN_MULTIPLE_SEEDS:
    # run with multiple seeds, show statistics and averaged graphs
    xAxis, yTest, dydxTest, stats, avg_values, avg_deltas = \
        run_digital_experiment_with_seeds(generator, sizes, nTest, NUM_SEEDS, weightSeed)

    # show statistics
    print_statistics(stats, sizes)

    # show averaged predictions
    graph_digital("Digital Options (averaged over %d seeds)" % NUM_SEEDS,
                  avg_values, xAxis, "", "values", yTest, sizes, True)

    if showDeltas:
        graph_digital("Digital Options (averaged over %d seeds)" % NUM_SEEDS,
                      avg_deltas, xAxis, "", "deltas", dydxTest, sizes, True)

else:
    # single run with graphs
    # simulSeed = 1234
    simulSeed = np.random.randint(0, 10000)
    xAxis, yTest, dydxTest, vegas, values, deltas = \
        test_digital(generator, sizes, nTest, simulSeed, None, weightSeed)

    print("using seed %d" % simulSeed)

    # show predicitions
    graph_digital("Digital Options", values, xAxis, "", "values", yTest, sizes, True)

    # show deltas
    if showDeltas:
        graph_digital("Digital Options", deltas, xAxis, "", "deltas", dydxTest, sizes, True)

## Digital Basket Options

Digital basket options with corrected implementation:
1. Fixed basket volatility (25%)
2. Pathwise delta = 0 (mathematically correct for digital payoffs)
3. LRM delta (theoretically correct for discontinuous payoffs)

We test across multiple dimensions (1, 7, 20) and correlation structures.

In [ ]:
# analytical formulas for digital Bachelier basket options
def digitalBachelierPrice(basket_value, strike, basket_vol, T):
    d = (basket_value - strike) / (basket_vol * np.sqrt(T))
    return norm.cdf(d)

def digitalBachelierDelta(basket_value, strike, basket_vol, T, weights):
    d = (basket_value - strike) / (basket_vol * np.sqrt(T))
    delta_factor = norm.pdf(d) / (basket_vol * np.sqrt(T))
    return weights * delta_factor

class BachelierBasketDigitalCorrected:

    def __init__(self, n, T1=1.0, T2=2.0, K=1.10, volMult=1.5, bktVol=BASKET_DIGITAL_VOL,
                 num_paths=BASKET_DIGITAL_NUM_PATHS, correlation=BASKET_DIGITAL_CORRELATION, seed=None):
        self.n = n
        self.T1 = T1
        self.T2 = T2
        self.dt = T2 - T1
        self.K = K
        self.volMult = volMult
        self.bktVol = bktVol
        self.num_paths = num_paths
        self.correlation = correlation
        self.rng = np.random.RandomState(seed)

        # S0 all ones
        self.S0 = np.ones(self.n)

        # equal weights
        self.a = np.ones(self.n) / self.n

        # pairwise correlation matrix
        self.corr = np.full((self.n, self.n), self.correlation)
        np.fill_diagonal(self.corr, 1.0)

        # fixed volatility implementation
        if self.n == 1:
            self.vols = np.array([self.bktVol])
        else:
            base_vol = self.bktVol / np.sqrt(1 + (self.n - 1) * self.correlation)
            self.vols = np.full(self.n, base_vol)

            # verify and adjust
            vol_matrix = np.outer(self.vols, self.vols)
            cov0 = vol_matrix * self.corr
            var_bkt0 = float(self.a.T @ cov0 @ self.a)
            actual_vol = np.sqrt(var_bkt0)

            # scale to match target
            scale = self.bktVol / actual_vol if actual_vol > 0 else 1.0
            self.vols = self.vols * scale

        # covariance matrix
        vol_matrix_scaled = np.outer(self.vols, self.vols)
        self.cov = vol_matrix_scaled * self.corr
        self.cov_inv = np.linalg.inv(self.cov + np.eye(self.n) * 1e-12)

        # Cholesky
        self.chol = np.linalg.cholesky(self.cov) * np.sqrt(self.dt)
        self.chol0 = np.linalg.cholesky(self.cov) * self.volMult * np.sqrt(self.T1)

    def trainingSet(self, m, seed=None):
        rng = np.random.RandomState(seed)

        # sample S1
        normals0 = rng.normal(size=(m, self.n))
        inc0 = normals0 @ self.chol0.T
        S1 = self.S0 + inc0

        # for each S1, average over num_paths draws for S2
        y_prices = np.zeros((m, 1))
        dydx_pathwise_corrected = np.zeros((m, self.n))  # pathwise = 0
        dydx_lrm = np.zeros((m, self.n))

        for i in range(m):
            normals1 = rng.normal(size=(self.num_paths, self.n))
            inc1 = normals1 @ self.chol.T
            S2 = S1[i:i+1, :] + inc1
            baskets = (S2 @ self.a.reshape(-1, 1)).reshape(-1)
            indicators = (baskets > self.K).astype(np.float64)

            # price: average binary payoff
            y_prices[i, 0] = indicators.mean()

            # pathwise delta = 0 for digital options
            dydx_pathwise_corrected[i, :] = 0.0

            # LRM: unchanged
            reg_param = 1e-8
            cov_reg = self.cov + np.eye(self.n) * reg_param * np.trace(self.cov) / self.n
            cov_inv_stable = np.linalg.inv(cov_reg)
            scores = (inc1 @ cov_inv_stable) / np.sqrt(self.dt)
            dydx_lrm[i, :] = (indicators.reshape(-1, 1) * scores).mean(axis=0)

        return S1, y_prices, dydx_pathwise_corrected, dydx_lrm

    def testSet(self, num=4096, lower=0.5, upper=1.5, seed=None):
        rng = np.random.RandomState(seed)

        # draw S1 uniformly in widened range
        adj = 1 + 0.5 * np.sqrt(max(0.0, (self.n - 1) * (upper - lower) / 12.0))
        adj_lower = 1.0 - (1.0 - lower) * adj
        adj_upper = 1.0 + (upper - 1.0) * adj
        S1 = rng.uniform(low=adj_lower, high=adj_upper, size=(num, self.n))

        # compute basket values
        baskets = (S1 @ self.a.reshape(-1, 1)).reshape(-1)

        # analytical prices
        prices = digitalBachelierPrice(baskets, self.K, self.bktVol, self.dt).reshape(-1, 1)

        # analytical deltas
        deltas = np.zeros((num, self.n))
        for i in range(num):
            deltas[i, :] = digitalBachelierDelta(baskets[i], self.K, self.bktVol, self.dt, self.a)

        vegas = np.zeros_like(prices)
        return S1, baskets, prices, deltas, vegas

# test function for digital baskets with 3 methods (standard, pathwise, lrm)
def test_digital_basket(generator,
                        sizes,
                        nTest,
                        simulSeed=None,
                        testSeed=None,
                        weightSeed=None,
                        deltidx=0):

    # simulation
    xTrain, yTrain, dydxTrain_pathwise, dydxTrain_lrm = \
        generator.trainingSet(max(sizes), seed=simulSeed)

    xTest, xAxis, yTest, dydxTest, vegas = \
        generator.testSet(num=nTest, seed=testSeed)

    # neural approximators - standard, pathwise (delta=0), and lrm
    regressor_standard = Neural_Approximator(xTrain, yTrain, None)
    regressor_pathwise = Neural_Approximator(xTrain, yTrain, dydxTrain_pathwise)
    regressor_lrm = Neural_Approximator(xTrain, yTrain, dydxTrain_lrm)

    predvalues = {}
    preddeltas = {}

    for size in sizes:

        # 1. Standard ML
        regressor_standard.prepare(size, False, weight_seed=weightSeed)
        regressor_standard.train("standard training")
        predictions, deltas = regressor_standard.predict_values_and_derivs(xTest)
        predvalues[("standard", size)] = predictions
        preddeltas[("standard", size)] = deltas[:, deltidx]

        # 2. Differential ML with pathwise derivatives (delta=0)
        regressor_pathwise.prepare(size, True, weight_seed=weightSeed)
        regressor_pathwise.train("pathwise training")
        predictions, deltas = regressor_pathwise.predict_values_and_derivs(xTest)
        predvalues[("pathwise", size)] = predictions
        preddeltas[("pathwise", size)] = deltas[:, deltidx]

        # 3. Differential ML with LRM
        regressor_lrm.prepare(size, True, weight_seed=weightSeed)
        regressor_lrm.train("lrm training")
        predictions, deltas = regressor_lrm.predict_values_and_derivs(xTest)
        predvalues[("lrm", size)] = predictions
        preddeltas[("lrm", size)] = deltas[:, deltidx]

    return xAxis, yTest, dydxTest[:, deltidx], vegas, predvalues, preddeltas

# graph function for digital baskets with 3 columns
def graph_digital_basket(title,
                         predictions,
                         xAxis,
                         xAxisName,
                         yAxisName,
                         targets,
                         sizes,
                         computeRmse=False,
                         weights=None):

    numRows = len(sizes)
    numCols = 3

    fig, ax = plt.subplots(numRows, numCols, squeeze=False)
    fig.set_size_inches(4 * numCols + 1.5, 4 * numRows)

    for i, size in enumerate(sizes):
        ax[i,0].annotate("size %d" % size, xy=(0, 0.5),
          xytext=(-ax[i,0].yaxis.labelpad-5, 0),
          xycoords=ax[i,0].yaxis.label, textcoords='offset points',
          ha='right', va='center')

    ax[0,0].set_title("standard")
    ax[0,1].set_title("pathwise")
    ax[0,2].set_title("lrm")

    for i, size in enumerate(sizes):
        for j, regType, in enumerate(["standard", "pathwise", "lrm"]):

            if computeRmse:
                errors = 100 * (predictions[(regType, size)] - targets)
                if weights is not None:
                    errors /= weights
                rmse = np.sqrt((errors ** 2).mean(axis=0))
                t = "rmse %.2f" % rmse
            else:
                t = xAxisName

            ax[i,j].set_xlabel(t)
            ax[i,j].set_ylabel(yAxisName)

            ax[i,j].plot(xAxis*100, predictions[(regType, size)]*100, 'co', \
                         markersize=2, markerfacecolor='white', label="predicted")
            ax[i,j].plot(xAxis*100, targets*100, 'o', color='darkred', markersize=1.5, label='targets')

            ax[i,j].legend(prop={'size': 8}, loc='upper left')

    plt.tight_layout()
    plt.subplots_adjust(top=0.9)
    plt.suptitle("% s -- %s" % (title, yAxisName), fontsize=16)
    plt.show()

In [ ]:
def run_digital_basket_experiment_with_seeds(generator, sizes, nTest, num_seeds=100, weightSeed=None):
    """
    Run digital basket experiment multiple times with different seeds

    Returns averaged predictions and statistics across all seeds
    """

    # storage for predictions and RMSE values across seeds
    all_predictions_values = {method: {size: [] for size in sizes} for method in ["standard", "pathwise", "lrm"]}
    all_predictions_deltas = {method: {size: [] for size in sizes} for method in ["standard", "pathwise", "lrm"]}
    rmse_values = {method: {size: [] for size in sizes} for method in ["standard", "pathwise", "lrm"]}
    rmse_deltas = {method: {size: [] for size in sizes} for method in ["standard", "pathwise", "lrm"]}

    # run experiment for each seed with progress bar
    for seed_idx in tqdm_notebook(range(num_seeds), desc="running seeds"):
        simulSeed = np.random.randint(0, 10000)

        # run experiment - FIXED: use fixed testSeed=42 so test set is same across all seeds
        xAxis, yTest, dydxTest, vegas, values, deltas = \
            test_digital_basket(generator, sizes, nTest, simulSeed, 42, weightSeed)

        # store predictions and compute RMSE for each method and size
        for method in ["standard", "pathwise", "lrm"]:
            for size in sizes:
                # store predictions
                all_predictions_values[method][size].append(values[(method, size)])
                all_predictions_deltas[method][size].append(deltas[(method, size)])

                # price RMSE
                rmse_val = np.sqrt(((values[(method, size)] - yTest)**2).mean())
                rmse_values[method][size].append(rmse_val)

                # delta RMSE
                rmse_delt = np.sqrt(((deltas[(method, size)] - dydxTest)**2).mean())
                rmse_deltas[method][size].append(rmse_delt)

    # compute averaged predictions
    avg_values = {}
    avg_deltas = {}
    for method in ["standard", "pathwise", "lrm"]:
        for size in sizes:
            avg_values[(method, size)] = np.mean(all_predictions_values[method][size], axis=0)
            avg_deltas[(method, size)] = np.mean(all_predictions_deltas[method][size], axis=0)

    # compute statistics
    stats = {}
    for metric, data in [("price", rmse_values), ("delta", rmse_deltas)]:
        stats[metric] = {}
        for method in ["standard", "pathwise", "lrm"]:
            stats[metric][method] = {}
            for size in sizes:
                values_array = np.array(data[method][size])
                stats[metric][method][size] = {
                    'mean': values_array.mean(),
                    'std': values_array.std(),
                    'min': values_array.min(),
                    'max': values_array.max()
                }

    return xAxis, yTest, dydxTest, stats, avg_values, avg_deltas

def print_basket_statistics(stats, sizes):
    """Print formatted statistics for multiple seed runs"""

    for metric in ["price", "delta"]:
        print(f"\n{metric.upper()} RMSE (over {NUM_SEEDS} seeds):")
        print(f"{'Method':<15} {'Size':<10} {'Mean':<12} {'Std':<12} {'Min':<12} {'Max':<12}")
        print("-"*70)

        for method in ["standard", "pathwise", "lrm"]:
            for size in sizes:
                s = stats[metric][method][size]
                print(f"{method:<15} {size:<10} {s['mean']:<12.6f} {s['std']:<12.6f} {s['min']:<12.6f} {s['max']:<12.6f}")

The code below trains two approximators for digital basket options. Standard ML and LRM ML
are compared across multiple dimensions.

In [ ]:
%matplotlib inline
# dimension 1
basketDim = 1
sizes = BASKET_DIGITAL_DIM1_SAMPLE_SIZES
nTest = BASKET_DIGITAL_DIM1_NUM_TEST
showDeltas = True
deltidx = 0
weightSeed = WEIGHT_SEED

generator = BachelierBasketDigitalCorrected(basketDim, correlation=BASKET_DIGITAL_CORRELATION)

if RUN_MULTIPLE_SEEDS:
    # run with multiple seeds, show statistics and averaged graphs
    xAxis, yTest, dydxTest, stats, avg_values, avg_deltas = \
        run_digital_basket_experiment_with_seeds(generator, sizes, nTest, NUM_SEEDS, weightSeed)

    # show statistics
    print_basket_statistics(stats, sizes)

    # show averaged predictions
    graph_digital_basket("Digital Basket dimension %d (averaged over %d seeds)" % (basketDim, NUM_SEEDS),
                  avg_values, xAxis, "", "values", yTest, sizes, True)

    if showDeltas:
        graph_digital_basket("Digital Basket dimension %d (averaged over %d seeds)" % (basketDim, NUM_SEEDS),
                      avg_deltas, xAxis, "", "deltas", dydxTest, sizes, True)

else:
    # single run with graphs
    simulSeed = np.random.randint(0, 10000)
    print("using seed %d" % simulSeed)

    xAxis, yTest, dydxTest, vegas, values, deltas = \
        test_digital_basket(generator, sizes, nTest, simulSeed, None, weightSeed, deltidx)

    # show predicitions
    graph_digital_basket("Digital Basket dimension %d" % basketDim, values, xAxis, "", "values", yTest, sizes, True)

    # show deltas
    if showDeltas:
        graph_digital_basket("Digital Basket dimension %d" % basketDim, deltas, xAxis, "", "deltas", dydxTest, sizes, True)

In [ ]:
%matplotlib inline
# dimension 7
basketDim = 7
sizes = BASKET_DIGITAL_DIM7_SAMPLE_SIZES
nTest = BASKET_DIGITAL_DIM7_NUM_TEST

generator = BachelierBasketDigitalCorrected(basketDim, correlation=BASKET_DIGITAL_CORRELATION)

if RUN_MULTIPLE_SEEDS:
    xAxis, yTest, dydxTest, stats, avg_values, avg_deltas = \
        run_digital_basket_experiment_with_seeds(generator, sizes, nTest, NUM_SEEDS, weightSeed)

    print_basket_statistics(stats, sizes)

    graph_digital_basket("Digital Basket dimension %d (averaged over %d seeds)" % (basketDim, NUM_SEEDS),
                  avg_values, xAxis, "", "values", yTest, sizes, True)

    if showDeltas:
        graph_digital_basket("Digital Basket dimension %d (averaged over %d seeds)" % (basketDim, NUM_SEEDS),
                      avg_deltas, xAxis, "", "deltas", dydxTest, sizes, True)

else:
    simulSeed = np.random.randint(0, 10000)
    print("using seed %d" % simulSeed)

    xAxis, yTest, dydxTest, vegas, values, deltas = \
        test_digital_basket(generator, sizes, nTest, simulSeed, None, weightSeed, deltidx)

    graph_digital_basket("Digital Basket dimension %d" % basketDim, values, xAxis, "", "values", yTest, sizes, True)

    if showDeltas:
        graph_digital_basket("Digital Basket dimension %d" % basketDim, deltas, xAxis, "", "deltas", dydxTest, sizes, True)

In [ ]:
%matplotlib inline
# dimension 20
basketDim = 20
sizes = BASKET_DIGITAL_DIM20_SAMPLE_SIZES
nTest = BASKET_DIGITAL_DIM20_NUM_TEST
showDeltas = True
deltidx = 0
weightSeed = WEIGHT_SEED

generator = BachelierBasketDigitalCorrected(basketDim, correlation=BASKET_DIGITAL_CORRELATION)

if RUN_MULTIPLE_SEEDS:
    xAxis, yTest, dydxTest, stats, avg_values, avg_deltas = \
        run_digital_basket_experiment_with_seeds(generator, sizes, nTest, NUM_SEEDS, weightSeed)

    print_basket_statistics(stats, sizes)

    graph_digital_basket("Digital Basket dimension %d (averaged over %d seeds)" % (basketDim, NUM_SEEDS),
                  avg_values, xAxis, "", "values", yTest, sizes, True)

    if showDeltas:
        graph_digital_basket("Digital Basket dimension %d (averaged over %d seeds)" % (basketDim, NUM_SEEDS),
                      avg_deltas, xAxis, "", "deltas", dydxTest, sizes, True)

else:
    simulSeed = np.random.randint(0, 10000)
    print("using seed %d" % simulSeed)

    xAxis, yTest, dydxTest, vegas, values, deltas = \
        test_digital_basket(generator, sizes, nTest, simulSeed, None, weightSeed, deltidx)

    graph_digital_basket("Digital Basket dimension %d" % basketDim, values, xAxis, "", "values", yTest, sizes, True)

    if showDeltas:
        graph_digital_basket("Digital Basket dimension %d" % basketDim, deltas, xAxis, "", "deltas", dydxTest, sizes, True)

## Barrier Options

Knockout barrier options with comparison of three methods:
1. Standard ML (no derivatives)
2. Pathwise ML (full payoff derivatives over T1→T3)
3. LRM ML (barrier survival derivatives over T1→T2)

Barrier structure: Down-and-out option checked at T2.

In [ ]:
class KnockoutBarrierOption:
    """
    Knockout barrier option simulator with:
    - Down-and-out barrier checked at T2
    - Pathwise derivatives over T1→T3 (full payoff)
    - LRM derivatives over T1→T2 (barrier survival)
    - Configurable parameters for survival rate calibration
    """

    def __init__(self, spot=BARRIER_SPOT, K=BARRIER_K, barrier=BARRIER_BARRIER, vol=BARRIER_VOL,
                 T1=BARRIER_T1, T2=BARRIER_T2, T3=BARRIER_T3, volMult=BARRIER_VOL_MULT):
        self.spot = spot
        self.K = K
        self.barrier = barrier
        self.vol = vol
        self.T1 = T1
        self.T2 = T2
        self.T3 = T3
        self.volMult = volMult

    def multipath_training_set(self, m, num_paths=100, seed=None):
        """
        Generate multi-path training data with both pathwise and LRM derivatives

        Returns:
            X: S1 values (m, 1)
            Y: Average payoffs (m, 1)
            Z_pathwise: Pathwise derivatives over T1→T3 (m, 1)
            Z_lrm: LRM derivatives over T1→T2 (m, 1)
        """
        np.random.seed(seed)

        # Generate S1 values
        returns1 = np.random.normal(size=[m, 1])
        vol0 = self.vol * self.volMult
        R1 = np.exp(-0.5*vol0*vol0*self.T1 + vol0*np.sqrt(self.T1)*returns1[:,0])
        S1 = self.spot * R1

        all_prices = []
        all_pathwise_derivs = []
        all_lrm_derivs = []

        for i in range(m):
            # For each S1, generate paths to S2 (barrier check)
            returns2 = np.random.normal(size=[num_paths, 1])
            Z_values_1 = returns2[:, 0]  # Z values for T1→T2 path

            R2_paths = np.exp(-0.5*self.vol*self.vol*(self.T2-self.T1) +
                            self.vol*np.sqrt(self.T2-self.T1)*returns2[:,0])
            S2_paths = S1[i] * R2_paths

            # Check barrier condition (down-and-out: survive if S2 >= barrier)
            barrier_condition = S2_paths >= self.barrier

            # For paths that survive the barrier, continue to S3
            path_payoffs = np.zeros_like(S2_paths)
            path_pathwise_derivs = np.zeros_like(S2_paths)
            path_lrm_derivs = np.zeros_like(S2_paths)

            for j in range(num_paths):
                if barrier_condition[j]:
                    # Generate path from S2 to S3
                    returns3 = np.random.normal(size=1)
                    R3 = np.exp(-0.5*self.vol*self.vol*(self.T3-self.T2) +
                                self.vol*np.sqrt(self.T3-self.T2)*returns3[0])
                    S3 = S2_paths[j] * R3

                    # Calculate payoff
                    payoff = max(0, S3 - self.K)
                    path_payoffs[j] = payoff

                    # Pathwise derivative over T1→T3: indicator(S3>K) * R3 * R2
                    indicator_S3 = 1.0 if S3 > self.K else 0.0
                    path_pathwise_derivs[j] = indicator_S3 * R3 * R2_paths[j]

                    # LRM derivative over T1→T2: payoff * Z / (S1 * σ * √T)
                    score1 = Z_values_1[j] / (S1[i] * self.vol * np.sqrt(self.T2-self.T1))
                    path_lrm_derivs[j] = payoff * score1

            # Average across paths
            avg_price = np.mean(path_payoffs)
            avg_pathwise_deriv = np.mean(path_pathwise_derivs)
            avg_lrm_deriv = np.mean(path_lrm_derivs)

            all_prices.append(avg_price)
            all_pathwise_derivs.append(avg_pathwise_deriv)
            all_lrm_derivs.append(avg_lrm_deriv)

        X = S1.reshape([-1, 1])
        Y = np.array(all_prices).reshape([-1, 1])
        Z_pathwise = np.array(all_pathwise_derivs).reshape([-1, 1])
        Z_lrm = np.array(all_lrm_derivs).reshape([-1, 1])

        return X, Y, Z_pathwise, Z_lrm

def _monte_carlo_price_barrier(barrier_model, S1_value, num_mc_paths, seed):
    """Helper function to compute Monte Carlo price for a single S1 value"""
    np.random.seed(seed)

    # Generate paths to S2 (barrier check)
    returns2 = np.random.normal(size=num_mc_paths)
    R2_paths = np.exp(-0.5*barrier_model.vol*barrier_model.vol*(barrier_model.T2-barrier_model.T1) +
                    barrier_model.vol*np.sqrt(barrier_model.T2-barrier_model.T1)*returns2)
    S2_paths = S1_value * R2_paths

    # Check barrier condition
    barrier_condition = S2_paths >= barrier_model.barrier

    # For paths that survive the barrier, continue to S3
    path_payoffs = np.zeros(num_mc_paths)

    for j in range(num_mc_paths):
        if barrier_condition[j]:
            # Generate path from S2 to S3
            returns3 = np.random.normal(size=1)
            R3 = np.exp(-0.5*barrier_model.vol*barrier_model.vol*(barrier_model.T3-barrier_model.T2) +
                        barrier_model.vol*np.sqrt(barrier_model.T3-barrier_model.T2)*returns3[0])
            S3 = S2_paths[j] * R3

            # Calculate payoff
            payoff = max(0, S3 - barrier_model.K)
            path_payoffs[j] = payoff

    return np.mean(path_payoffs)

def generate_barrier_ground_truth(barrier_model, lower=0.35, upper=1.65, num=100,
                                 num_mc_paths=1000000, finite_diff_bump=0.01, seed=None):
    """
    Generate high-accuracy ground truth with prices AND deltas using finite difference

    Returns:
        spots: Test S1 values
        prices: High-accuracy option prices
        deltas: Finite difference deltas (central difference)
        survival_rate: Estimated survival rate in test set
    """
    np.random.seed(seed)
    spots = np.linspace(lower, upper, num).reshape((-1, 1))
    prices = []
    deltas = []

    for i, spot in tqdm_notebook(enumerate(spots.flatten()), total=num, desc="generating ground truth"):
        # Central difference for delta: (f(S+ε) - f(S-ε)) / (2ε)
        price_plus = _monte_carlo_price_barrier(barrier_model, spot + finite_diff_bump, num_mc_paths, seed)
        price_minus = _monte_carlo_price_barrier(barrier_model, spot - finite_diff_bump, num_mc_paths, seed)
        price_center = _monte_carlo_price_barrier(barrier_model, spot, num_mc_paths, seed)

        delta_fd = (price_plus - price_minus) / (2 * finite_diff_bump)

        prices.append(price_center)
        deltas.append(delta_fd)

    prices = np.array(prices).reshape((-1, 1))
    deltas = np.array(deltas).reshape((-1, 1))

    # Calculate survival rate (non-zero payoffs indicate survival)
    survival_rate = np.mean(prices > 0.001)

    return spots, prices, deltas, survival_rate

def test_barrier(generator,
                 sizes,
                 nTest,
                 simulSeed=None,
                 testSeed=None,
                 weightSeed=None):
    """Test function for barrier options with 3 methods"""

    # Generate test set (ground truth with finite differences)
    xTest, yTest, dydxTest, survival_rate = generate_barrier_ground_truth(
        generator, num=nTest, num_mc_paths=GROUND_TRUTH_PATHS,
        finite_diff_bump=FINITE_DIFF_BUMP, seed=testSeed)

    # Generate training data
    xTrain, yTrain, dydxTrain_pathwise, dydxTrain_lrm = \
        generator.multipath_training_set(max(sizes), num_paths=BARRIER_NUM_PATHS_PER_S1, seed=simulSeed)

    # Neural approximators for all three methods
    regressor_standard = Neural_Approximator(xTrain, yTrain, None)
    regressor_pathwise = Neural_Approximator(xTrain, yTrain, dydxTrain_pathwise)
    regressor_lrm = Neural_Approximator(xTrain, yTrain, dydxTrain_lrm)

    predvalues = {}
    preddeltas = {}

    for size in sizes:

        # 1. Standard ML
        regressor_standard.prepare(size, False, weight_seed=weightSeed)
        regressor_standard.train("standard training")
        predictions, deltas = regressor_standard.predict_values_and_derivs(xTest)
        predvalues[("standard", size)] = predictions
        preddeltas[("standard", size)] = deltas[:, 0]

        # 2. Pathwise ML
        regressor_pathwise.prepare(size, True, weight_seed=weightSeed)
        regressor_pathwise.train("pathwise training")
        predictions, deltas = regressor_pathwise.predict_values_and_derivs(xTest)
        predvalues[("pathwise", size)] = predictions
        preddeltas[("pathwise", size)] = deltas[:, 0]

        # 3. LRM ML
        regressor_lrm.prepare(size, True, weight_seed=weightSeed)
        regressor_lrm.train("lrm training")
        predictions, deltas = regressor_lrm.predict_values_and_derivs(xTest)
        predvalues[("lrm", size)] = predictions
        preddeltas[("lrm", size)] = deltas[:, 0]

    return xTest, yTest, dydxTest, survival_rate, predvalues, preddeltas

def graph_barrier(title,
                  predictions,
                  xAxis,
                  xAxisName,
                  yAxisName,
                  targets,
                  sizes,
                  computeRmse=False,
                  weights=None):
    """Graph function for barrier options with 3 columns (standard, pathwise, lrm)"""

    numRows = len(sizes)
    numCols = 3

    fig, ax = plt.subplots(numRows, numCols, squeeze=False, figsize=(4 * numCols + 1.5, 4 * numRows))

    for i, size in enumerate(sizes):
        ax[i,0].annotate("size %d" % size, xy=(0, 0.5),
          xytext=(-ax[i,0].yaxis.labelpad-5, 0),
          xycoords=ax[i,0].yaxis.label, textcoords='offset points',
          ha='right', va='center')

    ax[0,0].set_title("standard")
    ax[0,1].set_title("pathwise")
    ax[0,2].set_title("lrm")

    for i, size in enumerate(sizes):
        for j, regType in enumerate(["standard", "pathwise", "lrm"]):

            if computeRmse:
                errors = predictions[(regType, size)].flatten() - targets.flatten()
                if weights is not None:
                    errors /= weights.flatten()
                rmse = float(np.sqrt((errors ** 2).mean()))
                t = "rmse %.4f" % rmse
            else:
                t = xAxisName

            ax[i,j].set_xlabel(t)
            ax[i,j].set_ylabel(yAxisName)

            ax[i,j].plot(xAxis.flatten(), predictions[(regType, size)].flatten(), 'co', \
                         markersize=2, markerfacecolor='white', label="predicted")
            ax[i,j].plot(xAxis.flatten(), targets.flatten(), 'o', color='darkred', markersize=1.5, label='targets')

            ax[i,j].legend(prop={'size': 8}, loc='upper left')

    plt.tight_layout()
    plt.subplots_adjust(top=0.9)
    plt.suptitle("%s -- %s" % (title, yAxisName), fontsize=16)
    plt.show()

In [ ]:
def rmse_barrier(predictions, targets):
    """Root Mean Square Error"""
    err = predictions - targets
    return float(np.sqrt(np.mean(err * err)))

def run_barrier_experiment_with_seeds(generator, sizes, nTest, num_seeds, weightSeed):
    """Run barrier experiment over multiple seeds and aggregate results"""

    # Generate test set once (fixed)
    xTest, yTest, dydxTest, survival_rate = generate_barrier_ground_truth(
        generator, num=nTest, num_mc_paths=GROUND_TRUTH_PATHS,
        finite_diff_bump=FINITE_DIFF_BUMP, seed=42)

    # Storage for results
    stats = {
        'price': {'standard': {}, 'pathwise': {}, 'lrm': {}},
        'delta': {'standard': {}, 'pathwise': {}, 'lrm': {}}
    }

    # Initialize storage for each size
    for size in sizes:
        for method in ['standard', 'pathwise', 'lrm']:
            stats['price'][method][size] = {'rmse': []}
            stats['delta'][method][size] = {'rmse': []}

    # Storage for averaged predictions
    avg_predictions = {method: {size: [] for size in sizes} for method in ['standard', 'pathwise', 'lrm']}
    avg_deltas = {method: {size: [] for size in sizes} for method in ['standard', 'pathwise', 'lrm']}

    # Run experiments
    for seed in tqdm_notebook(range(num_seeds), desc="barrier options"):

        # Reset graph
        # Generate training data
        xTrain, yTrain, dydxTrain_pathwise, dydxTrain_lrm = \
            generator.multipath_training_set(max(sizes), num_paths=BARRIER_NUM_PATHS_PER_S1, seed=seed+12345)

        # Neural approximators
        regressor_standard = Neural_Approximator(xTrain, yTrain, None)
        regressor_pathwise = Neural_Approximator(xTrain, yTrain, dydxTrain_pathwise)
        regressor_lrm = Neural_Approximator(xTrain, yTrain, dydxTrain_lrm)

        for size in sizes:
            # Standard ML
            regressor_standard.prepare(size, False, weight_seed=weightSeed)
            regressor_standard.train("standard")
            pred_vals, pred_deltas = regressor_standard.predict_values_and_derivs(xTest)

            stats['price']['standard'][size]['rmse'].append(rmse_barrier(pred_vals, yTest))
            stats['delta']['standard'][size]['rmse'].append(rmse_barrier(pred_deltas[:, 0:1], dydxTest))
            avg_predictions['standard'][size].append(pred_vals)
            avg_deltas['standard'][size].append(pred_deltas[:, 0:1])

            # Pathwise ML
            regressor_pathwise.prepare(size, True, weight_seed=weightSeed)
            regressor_pathwise.train("pathwise")
            pred_vals, pred_deltas = regressor_pathwise.predict_values_and_derivs(xTest)

            stats['price']['pathwise'][size]['rmse'].append(rmse_barrier(pred_vals, yTest))
            stats['delta']['pathwise'][size]['rmse'].append(rmse_barrier(pred_deltas[:, 0:1], dydxTest))
            avg_predictions['pathwise'][size].append(pred_vals)
            avg_deltas['pathwise'][size].append(pred_deltas[:, 0:1])

            # LRM ML
            regressor_lrm.prepare(size, True, weight_seed=weightSeed)
            regressor_lrm.train("lrm")
            pred_vals, pred_deltas = regressor_lrm.predict_values_and_derivs(xTest)

            stats['price']['lrm'][size]['rmse'].append(rmse_barrier(pred_vals, yTest))
            stats['delta']['lrm'][size]['rmse'].append(rmse_barrier(pred_deltas[:, 0:1], dydxTest))
            avg_predictions['lrm'][size].append(pred_vals)
            avg_deltas['lrm'][size].append(pred_deltas[:, 0:1])

    # Compute averaged predictions
    avg_values = {}
    avg_deltas_final = {}
    for method in ['standard', 'pathwise', 'lrm']:
        for size in sizes:
            avg_values[(method, size)] = np.mean(avg_predictions[method][size], axis=0)
            avg_deltas_final[(method, size)] = np.mean(avg_deltas[method][size], axis=0).flatten()

    # Compute statistics
    for metric in ['price', 'delta']:
        for method in ['standard', 'pathwise', 'lrm']:
            for size in sizes:
                values_array = np.array(stats[metric][method][size]['rmse'])
                stats[metric][method][size] = {
                    'mean': values_array.mean(),
                    'std': values_array.std(),
                    'min': values_array.min(),
                    'max': values_array.max()
                }

    return xTest, yTest, dydxTest, survival_rate, stats, avg_values, avg_deltas_final

def print_barrier_statistics(stats, sizes):
    """Print formatted statistics for barrier options"""

    for metric in ["price", "delta"]:
        print(f"\n{metric.upper()} RMSE (over {NUM_SEEDS} seeds):")
        print(f"{'Method':<15} {'Size':<10} {'Mean':<12} {'Std':<12} {'Min':<12} {'Max':<12}")
        print("-"*70)

        for method in ["standard", "pathwise", "lrm"]:
            for size in sizes:
                s = stats[metric][method][size]
                print(f"{method:<15} {size:<10} {s['mean']:<12.6f} {s['std']:<12.6f} {s['min']:<12.6f} {s['max']:<12.6f}")

Run barrier options experiment

In [ ]:
%matplotlib inline
sizes = BARRIER_SAMPLE_SIZES
nTest = BARRIER_NUM_TEST
showDeltas = True
weightSeed = WEIGHT_SEED

generator = KnockoutBarrierOption()

if RUN_MULTIPLE_SEEDS:
    # run with multiple seeds, show statistics and averaged graphs
    xAxis, yTest, dydxTest, survival_rate, stats, avg_values, avg_deltas = \
        run_barrier_experiment_with_seeds(generator, sizes, nTest, NUM_SEEDS, weightSeed)

    # show statistics
    print_barrier_statistics(stats, sizes)

    # show averaged predictions
    graph_barrier("Barrier Options (averaged over %d seeds)" % NUM_SEEDS,
                  avg_values, xAxis, "", "values", yTest, sizes, True)

    if showDeltas:
        graph_barrier("Barrier Options (averaged over %d seeds)" % NUM_SEEDS,
                      avg_deltas, xAxis, "", "deltas", dydxTest, sizes, True)

else:
    # single run with graphs
    simulSeed = np.random.randint(0, 10000)
    print("using seed %d" % simulSeed)

    xAxis, yTest, dydxTest, survival_rate, values, deltas = \
        test_barrier(generator, sizes, nTest, simulSeed, None, weightSeed)

    # show predictions
    graph_barrier("Barrier Options", values, xAxis, "", "values", yTest, sizes, True)

    # show deltas
    if showDeltas:
        graph_barrier("Barrier Options", deltas, xAxis, "", "deltas", dydxTest, sizes, True)

# Part III - Gamma Options (PW-LR Method)

Second-order derivatives using PW-LR (Pathwise-LRM) method for standard European calls.

Three methods compared:
1. Standard ML (price only)
2. Differential ML (price + pathwise delta)
3. PW-LR ML (price + pathwise delta + PW-LR gamma)

PW-LR Gamma Formula: e^{-rT} 1{S(T) > K} * (S(T)/S(0)^2) * (Z/(σ√T) - 1)
This applies LRM to the pathwise delta estimator to generate gamma labels.

In [ ]:
# Add PW-LR gamma generation method to existing BlackScholes class
def blackscholes_pw_lr_gamma_training_set(bs_model, m, num_paths=10, seed=None):
    """
    Generate training data with pathwise delta and PW-LR gamma

    Returns:
        X: S1 values (m, 1)
        Y: Average payoffs (m, 1)
        Z_pathwise: Pathwise delta (m, 1)
        G_pw_lr: PW-LR gamma (m, 1)
    """
    np.random.seed(seed)
    returns1 = np.random.normal(size=[m, 1])
    vol0 = bs_model.vol * bs_model.volMult
    R1 = np.exp(-0.5*vol0*vol0*bs_model.T1 + vol0*np.sqrt(bs_model.T1)*returns1[:,0])
    S1 = bs_model.spot * R1

    all_prices = []
    all_pathwise_delta = []
    all_pw_lr_gamma = []
    dt = (bs_model.T2 - bs_model.T1)

    for i in range(m):
        returns2 = np.random.normal(size=[num_paths, 1])
        Z = returns2[:, 0]  # Random variables for LRM
        R2 = np.exp(-0.5*bs_model.vol*bs_model.vol*dt + bs_model.vol*np.sqrt(dt)*returns2[:,0])
        S2 = S1[i] * R2

        # Option payoffs
        payoffs = np.maximum(S2 - bs_model.K, 0.0)
        indicator = (S2 > bs_model.K).astype(float)

        # Pathwise delta (standard formula)
        delta_pw = indicator * R2

        # PW-LR Gamma: e^{-rT} 1{S(T) > K} * (S(T)/S(0)^2) * (Z/(σ√T) - 1)
        gamma_pw_lr = indicator * (S2 / (S1[i]**2)) * (Z / (bs_model.vol * np.sqrt(dt)) - 1.0)

        # Average across paths
        all_prices.append(np.mean(payoffs))
        all_pathwise_delta.append(np.mean(delta_pw))
        all_pw_lr_gamma.append(np.mean(gamma_pw_lr))

    X = S1.reshape([-1, 1])
    Y = np.array(all_prices).reshape([-1, 1])
    Z_pathwise = np.array(all_pathwise_delta).reshape([-1, 1])
    G_pw_lr = np.array(all_pw_lr_gamma).reshape([-1, 1])

    return X, Y, Z_pathwise, G_pw_lr

# Note: Part I already has all gamma infrastructure (second_order_twin_net, gamma_training_graph, etc.)
# Using existing Part I Neural_Approximator with use_gamma=True for tri-objective training

def test_gamma(generator,
               sizes,
               nTest,
               simulSeed=None,
               testSeed=None,
               weightSeed=None):
    """Test function for gamma options with 3 methods"""

    # Generate test set (analytical formulas)
    xTest, xAxis, yTest, dydxTest, d2ydx2Test = generator.testSet(num=nTest, seed=testSeed)

    # Generate training data with PW-LR gamma
    xTrain, yTrain, dydxTrain, d2ydx2Train = \
        blackscholes_pw_lr_gamma_training_set(generator, max(sizes), num_paths=GAMMA_NUM_PATHS_PER_S1, seed=simulSeed)

    # Neural approximators for all three methods (using existing Part I Neural_Approximator)
    regressor_standard = Neural_Approximator(xTrain, yTrain, None)
    regressor_diff = Neural_Approximator(xTrain, yTrain, dydxTrain)
    regressor_pw_lr = Neural_Approximator(xTrain, yTrain, dydxTrain, d2ydx2Train)

    predvalues = {}
    preddeltas = {}
    predgammas = {}

    for size in sizes:

        # 1. Standard ML
        regressor_standard.prepare(size, False, weight_seed=weightSeed)
        regressor_standard.train("standard training")
        predictions = regressor_standard.predict_values(xTest)
        predvalues[("standard", size)] = predictions

        # 2. Differential ML
        regressor_diff.prepare(size, True, weight_seed=weightSeed)
        regressor_diff.train("differential training")
        predictions, deltas = regressor_diff.predict_values_and_derivs(xTest)
        predvalues[("differential", size)] = predictions
        preddeltas[("differential", size)] = deltas[:, 0]

        # 3. PW-LR ML (using existing gamma support in Part I)
        regressor_pw_lr.prepare(size, True, use_gamma=True, weight_seed=weightSeed)
        regressor_pw_lr.train("pw-lr training")
        predictions, deltas, gammas = regressor_pw_lr.predict_values_derivs_and_gamma(xTest)
        predvalues[("pw_lr", size)] = predictions
        preddeltas[("pw_lr", size)] = deltas[:, 0]
        predgammas[("pw_lr", size)] = gammas[:, 0]

    return xAxis, yTest, dydxTest, d2ydx2Test, predvalues, preddeltas, predgammas

def graph_gamma(title,
                predictions,
                xAxis,
                xAxisName,
                yAxisName,
                targets,
                sizes,
                computeRmse=False,
                methods=None):
    """Graph function for gamma options with dynamic columns based on available methods"""

    if methods is None:
        methods = ["standard", "differential", "pw_lr"]

    # Filter to only methods that have data
    available_methods = [m for m in methods if (m, sizes[0]) in predictions]

    numRows = len(sizes)
    numCols = len(available_methods)

    fig, ax = plt.subplots(numRows, numCols, squeeze=False, figsize=(4 * numCols + 1.5, 4 * numRows))

    for i, size in enumerate(sizes):
        ax[i,0].annotate("size %d" % size, xy=(0, 0.5),
          xytext=(-ax[i,0].yaxis.labelpad-5, 0),
          xycoords=ax[i,0].yaxis.label, textcoords='offset points',
          ha='right', va='center')

    for j, method in enumerate(available_methods):
        ax[0,j].set_title(method)

    for i, size in enumerate(sizes):
        for j, regType in enumerate(available_methods):

            if computeRmse:
                errors = predictions[(regType, size)].flatten() - targets.flatten()
                rmse = float(np.sqrt((errors ** 2).mean()))
                t = "rmse %.4f" % rmse
            else:
                t = xAxisName

            ax[i,j].set_xlabel(t)
            ax[i,j].set_ylabel(yAxisName)

            ax[i,j].plot(xAxis.flatten(), predictions[(regType, size)].flatten(), 'co', \
                         markersize=2, markerfacecolor='white', label="predicted")
            ax[i,j].plot(xAxis.flatten(), targets.flatten(), 'o', color='darkred', markersize=1.5, label='targets')

            ax[i,j].legend(prop={'size': 8}, loc='upper left')

    plt.tight_layout()
    plt.subplots_adjust(top=0.9)
    plt.suptitle("%s -- %s" % (title, yAxisName), fontsize=16)
    plt.show()

In [ ]:
def rmse_gamma(predictions, targets):
    """Root Mean Square Error"""
    err = predictions - targets
    return float(np.sqrt(np.mean(err * err)))

def run_gamma_experiment_with_seeds(generator, sizes, nTest, num_seeds, weightSeed):
    """Run gamma experiment over multiple seeds and aggregate results"""

    # Generate test set once (fixed)
    xTest, xAxis, yTest, dydxTest, d2ydx2Test = generator.testSet(num=nTest, seed=42)

    # Storage for results
    stats = {
        'price': {'standard': {}, 'differential': {}, 'pw_lr': {}},
        'delta': {'differential': {}, 'pw_lr': {}},
        'gamma': {'pw_lr': {}}
    }

    # Initialize storage for each size
    for size in sizes:
        stats['price']['standard'][size] = {'rmse': []}
        stats['price']['differential'][size] = {'rmse': []}
        stats['price']['pw_lr'][size] = {'rmse': []}
        stats['delta']['differential'][size] = {'rmse': []}
        stats['delta']['pw_lr'][size] = {'rmse': []}
        stats['gamma']['pw_lr'][size] = {'rmse': []}

    # Storage for averaged predictions
    avg_predictions = {method: {size: [] for size in sizes} for method in ['standard', 'differential', 'pw_lr']}
    avg_deltas = {method: {size: [] for size in sizes} for method in ['differential', 'pw_lr']}
    avg_gammas = {'pw_lr': {size: [] for size in sizes}}

    # Run experiments
    for seed in tqdm_notebook(range(num_seeds), desc="gamma options"):

        # Reset graph
        # Generate training data
        xTrain, yTrain, dydxTrain, d2ydx2Train = \
            blackscholes_pw_lr_gamma_training_set(generator, max(sizes), num_paths=GAMMA_NUM_PATHS_PER_S1, seed=seed+12345)

        # Neural approximators (using existing Part I Neural_Approximator)
        regressor_standard = Neural_Approximator(xTrain, yTrain, None)
        regressor_diff = Neural_Approximator(xTrain, yTrain, dydxTrain)
        regressor_pw_lr = Neural_Approximator(xTrain, yTrain, dydxTrain, d2ydx2Train)

        for size in sizes:
            # Standard ML
            regressor_standard.prepare(size, False, weight_seed=weightSeed)
            regressor_standard.train("standard")
            pred_vals = regressor_standard.predict_values(xTest)

            stats['price']['standard'][size]['rmse'].append(rmse_gamma(pred_vals, yTest))
            avg_predictions['standard'][size].append(pred_vals)

            # Differential ML
            regressor_diff.prepare(size, True, weight_seed=weightSeed)
            regressor_diff.train("differential")
            pred_vals, pred_deltas = regressor_diff.predict_values_and_derivs(xTest)

            stats['price']['differential'][size]['rmse'].append(rmse_gamma(pred_vals, yTest))
            stats['delta']['differential'][size]['rmse'].append(rmse_gamma(pred_deltas[:, 0:1], dydxTest[:, 0:1]))
            avg_predictions['differential'][size].append(pred_vals)
            avg_deltas['differential'][size].append(pred_deltas[:, 0:1])

            # PW-LR ML (using existing gamma support in Part I)
            regressor_pw_lr.prepare(size, True, use_gamma=True, weight_seed=weightSeed)
            regressor_pw_lr.train("pw-lr")
            pred_vals, pred_deltas, pred_gammas = regressor_pw_lr.predict_values_derivs_and_gamma(xTest)

            stats['price']['pw_lr'][size]['rmse'].append(rmse_gamma(pred_vals, yTest))
            stats['delta']['pw_lr'][size]['rmse'].append(rmse_gamma(pred_deltas[:, 0:1], dydxTest[:, 0:1]))
            stats['gamma']['pw_lr'][size]['rmse'].append(rmse_gamma(pred_gammas[:, 0:1], d2ydx2Test[:, 0:1]))
            avg_predictions['pw_lr'][size].append(pred_vals)
            avg_deltas['pw_lr'][size].append(pred_deltas[:, 0:1])
            avg_gammas['pw_lr'][size].append(pred_gammas[:, 0:1])

    # Compute averaged predictions
    avg_values = {}
    avg_deltas_final = {}
    avg_gammas_final = {}
    for size in sizes:
        avg_values[("standard", size)] = np.mean(avg_predictions['standard'][size], axis=0)
        avg_values[("differential", size)] = np.mean(avg_predictions['differential'][size], axis=0)
        avg_values[("pw_lr", size)] = np.mean(avg_predictions['pw_lr'][size], axis=0)
        avg_deltas_final[("differential", size)] = np.mean(avg_deltas['differential'][size], axis=0).flatten()
        avg_deltas_final[("pw_lr", size)] = np.mean(avg_deltas['pw_lr'][size], axis=0).flatten()
        avg_gammas_final[("pw_lr", size)] = np.mean(avg_gammas['pw_lr'][size], axis=0).flatten()

    # Compute statistics
    for metric in ['price', 'delta', 'gamma']:
        for method in stats[metric].keys():
            for size in sizes:
                if size in stats[metric][method]:
                    values_array = np.array(stats[metric][method][size]['rmse'])
                    stats[metric][method][size] = {
                        'mean': values_array.mean(),
                        'std': values_array.std(),
                        'min': values_array.min(),
                        'max': values_array.max()
                    }

    return xAxis, yTest, dydxTest, d2ydx2Test, stats, avg_values, avg_deltas_final, avg_gammas_final

def print_gamma_statistics(stats, sizes):
    """Print formatted statistics for gamma options"""

    print(f"\nPRICE RMSE (over {NUM_SEEDS} seeds):")
    print(f"{'Method':<15} {'Size':<10} {'Mean':<12} {'Std':<12} {'Min':<12} {'Max':<12}")
    print("-"*70)

    for method in ["standard", "differential", "pw_lr"]:
        for size in sizes:
            s = stats['price'][method][size]
            print(f"{method:<15} {size:<10} {s['mean']:<12.6f} {s['std']:<12.6f} {s['min']:<12.6f} {s['max']:<12.6f}")

    print(f"\nDELTA RMSE (over {NUM_SEEDS} seeds):")
    print(f"{'Method':<15} {'Size':<10} {'Mean':<12} {'Std':<12} {'Min':<12} {'Max':<12}")
    print("-"*70)

    for method in ["differential", "pw_lr"]:
        for size in sizes:
            s = stats['delta'][method][size]
            print(f"{method:<15} {size:<10} {s['mean']:<12.6f} {s['std']:<12.6f} {s['min']:<12.6f} {s['max']:<12.6f}")

    print(f"\nGAMMA RMSE (over {NUM_SEEDS} seeds):")
    print(f"{'Method':<15} {'Size':<10} {'Mean':<12} {'Std':<12} {'Min':<12} {'Max':<12}")
    print("-"*70)

    for size in sizes:
        s = stats['gamma']['pw_lr'][size]
        print(f"{'pw_lr':<15} {size:<10} {s['mean']:<12.6f} {s['std']:<12.6f} {s['min']:<12.6f} {s['max']:<12.6f}")

Run gamma options experiment

In [ ]:
%matplotlib inline

sizes = GAMMA_SAMPLE_SIZES
nTest = GAMMA_NUM_TEST
showDeltas = True
showGammas = True
weightSeed = WEIGHT_SEED

generator = BlackScholes()

if RUN_MULTIPLE_SEEDS:
    # run with multiple seeds, show statistics and averaged graphs
    xAxis, yTest, dydxTest, d2ydx2Test, stats, avg_values, avg_deltas, avg_gammas = \
        run_gamma_experiment_with_seeds(generator, sizes, nTest, NUM_SEEDS, weightSeed)

    # show statistics
    print_gamma_statistics(stats, sizes)

    # show averaged predictions - prices
    graph_gamma("Gamma Options (averaged over %d seeds)" % NUM_SEEDS,
                  avg_values, xAxis, "", "values", yTest, sizes, True)

    # show averaged predictions - deltas
    if showDeltas:
        graph_gamma("Gamma Options Deltas (averaged over %d seeds)" % NUM_SEEDS,
                      avg_deltas, xAxis, "", "deltas", dydxTest[:, 0:1], sizes, True)

    # show averaged predictions - gammas
    if showGammas:
        graph_gamma("Gamma Options Gammas (averaged over %d seeds)" % NUM_SEEDS,
                      avg_gammas, xAxis, "", "gammas", d2ydx2Test[:, 0:1], sizes, True)

else:
    # single run with graphs
    simulSeed = np.random.randint(0, 10000)
    print("using seed %d" % simulSeed)

    xAxis, yTest, dydxTest, d2ydx2Test, values, deltas, gammas = \
        test_gamma(generator, sizes, nTest, simulSeed, None, weightSeed)

    # show predictions - prices
    graph_gamma("Gamma Options", values, xAxis, "", "values", yTest, sizes, True)

    # show predictions - deltas
    if showDeltas:
        graph_gamma("Gamma Options", deltas, xAxis, "", "deltas", dydxTest[:, 0:1], sizes, True)

    # show predictions - gammas
    if showGammas:
        graph_gamma("Gamma Options", gammas, xAxis, "", "gammas", d2ydx2Test[:, 0:1], sizes, True)

# Part IV

## Ramp Smoothened Digital Options Experiment

This experiment demonstrates that even when we smooth the discontinuous digital payoff
to make pathwise derivatives well-defined, the LRM method still outperforms pathwise
differential ML.

### The Ramp Payoff

Instead of a step function at K, we use a ramp from K-ε to K+ε:
- Payoff = 0 if S < K-ε
- Payoff = (S - (K-ε))/(2ε) if K-ε ≤ S ≤ K+ε (linear ramp)
- Payoff = 1 if S > K+ε

This makes the payoff differentiable, so pathwise derivatives exist:
- Delta = 1/(2ε) in the ramp region, 0 elsewhere

### The Experiment

We compare three methods:
1. **Standard ML**: No derivatives
2. **Differential ML (Pathwise)**: Uses smoothed ramp payoff and its derivative
3. **Differential ML (LRM)**: Uses original step function (indicator)

We test different smoothing parameters: ε = {0.1, 0.5, 1.0, 2.0} × σ√T

The hypothesis: Even with smoothing to make pathwise "work", LRM still performs better
because it properly handles the discontinuity through the score function.

In [ ]:
class RampDigitalOption:
    """
    Digital option with ramped (smoothed) payoff for pathwise differential ML.

    The ramp creates a smooth transition from 0 to 1 around the strike, making
    the payoff differentiable so pathwise derivatives can be computed.
    """

    def __init__(self, spot=RAMP_SPOT, K=RAMP_STRIKE, vol=RAMP_VOLATILITY, T1=RAMP_T1, T2=RAMP_T2,
                 volMult=RAMP_VOL_MULT, epsilon_multiplier=1.0):
        self.spot = spot
        self.K = K
        self.vol = vol
        self.T1 = T1
        self.T2 = T2
        self.volMult = volMult

        # Smoothing parameter: epsilon = epsilon_multiplier * vol * sqrt(T2-T1)
        self.epsilon_mult = epsilon_multiplier
        self.epsilon = epsilon_multiplier * vol * np.sqrt(T2 - T1)

    def ramp_payoff(self, S):
        """
        Ramp payoff function: smooth transition from 0 to 1

        Returns:
            payoff: 0 if S < K-ε, linear ramp if K-ε ≤ S ≤ K+ε, 1 if S > K+ε
        """
        K_lower = self.K - self.epsilon
        K_upper = self.K + self.epsilon

        payoff = np.where(S < K_lower, 0.0,
                 np.where(S > K_upper, 1.0,
                         (S - K_lower) / (2 * self.epsilon)))
        return payoff

    def ramp_delta(self, S):
        """
        Delta of ramp payoff: derivative with respect to S

        Returns:
            delta: 1/(2ε) in ramp region, 0 elsewhere
        """
        K_lower = self.K - self.epsilon
        K_upper = self.K + self.epsilon

        delta = np.where((S >= K_lower) & (S <= K_upper),
                        1.0 / (2 * self.epsilon),
                        0.0)
        return delta

    def trainingSet(self, m, seed=None):
        """
        Generate training set with both ramp and digital payoffs

        Returns:
            X: S1 values (mx1)
            Y_digital: Digital (step) payoffs (mx1) - for standard ML and LRM
            Y_ramp: Ramp (smooth) payoffs (mx1) - for pathwise ML only
            Z_pathwise: Ramp deltas (mx1) - smooth, well-defined
            Z_lrm: LRM derivatives (mx1) - using step function indicator
        """
        np.random.seed(seed)

        # Generate returns
        returns = np.random.normal(size=[m, 2])

        # SDE
        vol0 = self.vol * self.volMult
        R1 = np.exp(-0.5*vol0*vol0*self.T1 + vol0*np.sqrt(self.T1)*returns[:,0])
        R2 = np.exp(-0.5*self.vol*self.vol*(self.T2-self.T1) +
                    self.vol*np.sqrt(self.T2-self.T1)*returns[:,1])
        S1 = self.spot * R1
        S2 = S1 * R2

        # Antithetic paths
        R2a = np.exp(-0.5*self.vol*self.vol*(self.T2-self.T1) -
                     self.vol*np.sqrt(self.T2-self.T1)*returns[:,1])
        S2a = S1 * R2a

        # Digital payoffs (for standard ML and LRM) - binary payoffs (TRUE digital option)
        pay_digital = np.where(S2 >= self.K, 1.0, 0.0)
        paya_digital = np.where(S2a >= self.K, 1.0, 0.0)

        # Ramp payoffs (for pathwise ML only)
        pay_ramp = self.ramp_payoff(S2)
        paya_ramp = self.ramp_payoff(S2a)

        X = S1
        Y_digital = 0.5 * (pay_digital + paya_digital)
        Y_ramp = 0.5 * (pay_ramp + paya_ramp)

        # Pathwise derivatives: delta of ramp payoff
        delta1 = self.ramp_delta(S2)
        delta2 = self.ramp_delta(S2a)
        Z_pathwise = 0.5 * (delta1 * R2 + delta2 * R2a)

        # LRM derivatives: using step function (original digital)
        Z_values1 = returns[:, 1]
        Z_values2 = -returns[:, 1]
        indicator1 = np.where(S2 >= self.K, 1.0, 0.0)
        indicator2 = np.where(S2a >= self.K, 1.0, 0.0)
        lrm1 = indicator1 * (Z_values1 / (S1 * self.vol * np.sqrt(self.T2-self.T1)))
        lrm2 = indicator2 * (Z_values2 / (S1 * self.vol * np.sqrt(self.T2-self.T1)))
        Z_lrm = 0.5 * (lrm1 + lrm2)

        return X.reshape([-1,1]), Y_digital.reshape([-1,1]), Y_ramp.reshape([-1,1]), \
               Z_pathwise.reshape([-1,1]), Z_lrm.reshape([-1,1])

    def testSet(self, lower=0.35, upper=1.65, num=100, seed=None):
        """
        Generate test set with analytical ground truth

        Returns TRUE DIGITAL ground truth (not ramp) for fair comparison.
        """
        np.random.seed(seed)
        spots = np.linspace(lower, upper, num).reshape((-1, 1))

        # Use analytical digital option formulas (TRUE ground truth)
        T = self.T2 - self.T1
        d2 = (np.log(spots/self.K) / (self.vol * np.sqrt(T))) - (self.vol * np.sqrt(T) / 2)

        # Digital option price and delta
        prices = norm.cdf(d2)
        deltas = norm.pdf(d2) / (spots * self.vol * np.sqrt(T))

        vegas = np.zeros_like(prices)
        xAxis = spots

        return spots, xAxis, prices, deltas, vegas

# Test function for ramp digital options
def test_ramp_digital(generator, sizes, nTest, simulSeed=None, testSeed=None,
                      weightSeed=None, deltidx=0):
    """
    Test ramp digital options with three methods:
    1. Standard ML - trains on digital payoffs
    2. Differential ML with pathwise (ramp) derivatives - trains on ramp payoffs
    3. Differential ML with LRM derivatives - trains on digital payoffs

    All test against TRUE DIGITAL ground truth.
    """

    # Generate training data
    xTrain, yTrain_digital, yTrain_ramp, dydxTrain_pathwise, dydxTrain_lrm = \
        generator.trainingSet(max(sizes), seed=simulSeed)

    xTest, xAxis, yTest, dydxTest, vegas = \
        generator.testSet(num=nTest, seed=testSeed)

    # Neural approximators - IMPORTANT: pathwise uses ramp payoffs, others use digital
    regressor_standard = Neural_Approximator(xTrain, yTrain_digital, None)
    regressor_pathwise = Neural_Approximator(xTrain, yTrain_ramp, dydxTrain_pathwise)
    regressor_lrm = Neural_Approximator(xTrain, yTrain_digital, dydxTrain_lrm)

    predvalues = {}
    preddeltas = {}

    for size in sizes:

        # 1. Standard ML
        regressor_standard.prepare(size, False, weight_seed=weightSeed)
        regressor_standard.train("standard training")
        predictions, deltas = regressor_standard.predict_values_and_derivs(xTest)
        predvalues[("standard", size)] = predictions
        preddeltas[("standard", size)] = deltas[:, deltidx]

        # 2. Differential ML with pathwise (ramp) derivatives
        regressor_pathwise.prepare(size, True, weight_seed=weightSeed)
        regressor_pathwise.train("pathwise training")
        predictions, deltas = regressor_pathwise.predict_values_and_derivs(xTest)
        predvalues[("pathwise", size)] = predictions
        preddeltas[("pathwise", size)] = deltas[:, deltidx]

        # 3. Differential ML with LRM
        regressor_lrm.prepare(size, True, weight_seed=weightSeed)
        regressor_lrm.train("lrm training")
        predictions, deltas = regressor_lrm.predict_values_and_derivs(xTest)
        predvalues[("lrm", size)] = predictions
        preddeltas[("lrm", size)] = deltas[:, deltidx]

    return xAxis, yTest, dydxTest[:, deltidx], vegas, predvalues, preddeltas

# Graph function for ramp digital options (3 columns: standard, pathwise, lrm)
def graph_ramp_digital(title, predictions, xAxis, xAxisName, yAxisName,
                       targets, sizes, computeRmse=False, weights=None):
    """
    Graph predictions for ramp digital options with 3 methods
    """

    numRows = len(sizes)
    numCols = 3

    fig, ax = plt.subplots(numRows, numCols, squeeze=False)
    fig.set_size_inches(4 * numCols + 1.5, 4 * numRows)

    for i, size in enumerate(sizes):
        ax[i,0].annotate("size %d" % size, xy=(0, 0.5),
          xytext=(-ax[i,0].yaxis.labelpad-5, 0),
          xycoords=ax[i,0].yaxis.label, textcoords='offset points',
          ha='right', va='center')

    ax[0,0].set_title("standard")
    ax[0,1].set_title("pathwise (ramp)")
    ax[0,2].set_title("lrm")

    for i, size in enumerate(sizes):
        for j, regType in enumerate(["standard", "pathwise", "lrm"]):

            if computeRmse:
                errors = 100 * (predictions[(regType, size)] - targets)
                if weights is not None:
                    errors /= weights
                rmse = np.sqrt((errors ** 2).mean(axis=0))
                t = "rmse %.2f" % rmse
            else:
                t = xAxisName

            ax[i,j].set_xlabel(t)
            ax[i,j].set_ylabel(yAxisName)

            ax[i,j].plot(xAxis*100, predictions[(regType, size)]*100, 'co',
                        markersize=2, markerfacecolor='white', markeredgewidth=0.5)
            ax[i,j].plot(xAxis*100, targets*100, 'r-', linewidth=1)

    plt.tight_layout()
    plt.subplots_adjust(top=0.9)
    plt.suptitle("%s -- %s" % (title, yAxisName), fontsize=16)
    plt.show()

## Ramp Smoothened Digital Options Experiments

We test different smoothing parameters ε to show that even with varying degrees
of smoothing, LRM consistently outperforms pathwise differential ML.


In [ ]:
%matplotlib inline

# Configuration for ramp digital experiments
# Use same configuration as standard digital options
sizes = BS_SAMPLE_SIZES
nTest = BS_NUM_TEST
EPSILON_MULTIPLIERS = [0.1, 0.5, 1.0, 2.0]  # Multiples of σ√T
showDeltas = True
weightSeed = WEIGHT_SEED

print("\n" + "="*80)
print("RAMP DIGITAL OPTIONS EXPERIMENTS")
print("="*80)
print("\nTesting epsilon values: ", [f"{em}×σ√T" for em in EPSILON_MULTIPLIERS])

for epsilon_mult in EPSILON_MULTIPLIERS:
    print(f"\n{'='*80}")
    print(f"Epsilon = {epsilon_mult}×σ√T")
    print(f"{'='*80}\n")

    generator = RampDigitalOption(epsilon_multiplier=epsilon_mult)

    simulSeed = np.random.randint(0, 10000)
    print("using seed %d" % simulSeed)

    xAxis, yTest, dydxTest, vegas, values, deltas = \
        test_ramp_digital(generator, sizes, nTest, simulSeed, None, weightSeed)

    # Show predictions - prices
    graph_ramp_digital(f"Ramp Digital (ε={epsilon_mult}×σ√T)",
                       values, xAxis, "", "values", yTest, sizes, True)

    # Show predictions - deltas
    if showDeltas:
        graph_ramp_digital(f"Ramp Digital (ε={epsilon_mult}×σ√T)",
                          deltas, xAxis, "", "deltas", dydxTest, sizes, True)

# PART V: Portfolio Options Experiment

This experiment tests ML methods on complex option portfolios:
1. Call Portfolio: C(0.85) - 1.5*C(0.9) + 0.75*C(1.15)
2. Digital Portfolio: D(0.75) - D(0.95) + D(1.15) - D(1.35)

These combinations create more complex payoff structures that are harder for
standard ML to learn, providing a rigorous test of differential machine learning methods.

Methods Compared:
- Call portfolio: Standard, Delta-Pathwise, Delta-LRM, PW-LR (tri-objective, 40-40-20 weights)
- Digital portfolio: Standard vs. Delta-LRM (PW-LR excluded because pathwise estimators are biased on digital payoffs)

Parameters:
- vol = 0.20
- T = 1/3
- Compare value/delta/gamma RMSE across methods

In [ ]:
class CallPortfolioOption:
    """Call portfolio: C(0.85) - 1.5*C(0.9) + 0.75*C(1.15)

    Uses Black-Scholes (geometric Brownian motion) to be consistent with
    other single-asset experiments in this notebook.
    """

    def __init__(self, vol=PORTFOLIO_VOL, T1=PORTFOLIO_T1, T2=PORTFOLIO_T2, strikes=None, weights=None,
                 volMult=PORTFOLIO_VOL_MULT, num_paths=PORTFOLIO_NUM_PATHS_PER_S1, seed=None):
        self.spot = PORTFOLIO_SPOT
        self.vol = vol
        self.T1 = T1
        self.T2 = T2
        self.dt = T2 - T1
        self.strikes = strikes if strikes is not None else PORTFOLIO_CALL_STRIKES
        self.weights = weights if weights is not None else PORTFOLIO_CALL_WEIGHTS
        self.volMult = volMult
        self.num_paths = num_paths
        self.rng = np.random.RandomState(seed)

    def trainingSet(self, m, seed=None):
        """Generate training data with pathwise delta, LRM delta, and PW-LR gamma
        Uses Black-Scholes (geometric Brownian motion)
        """
        rng = np.random.RandomState(seed)

        # Sample S1 with volMult for better coverage (geometric Brownian motion)
        returns0 = rng.normal(size=m)
        vol0 = self.vol * self.volMult
        R1 = np.exp(-0.5*vol0*vol0*self.T1 + vol0*np.sqrt(self.T1)*returns0)
        S1 = self.spot * R1

        # For each S1, average over num_paths draws for S2
        y_prices = np.zeros((m, 1))
        dydx_pathwise = np.zeros((m, 1))
        dydx_lrm = np.zeros((m, 1))
        d2ydx2_lrm = np.zeros((m, 1))

        for i in range(m):
            returns1 = rng.normal(size=self.num_paths)
            # Geometric Brownian motion: S2 = S1 * exp(-0.5*vol^2*dt + vol*sqrt(dt)*Z)
            R2 = np.exp(-0.5*self.vol*self.vol*self.dt + self.vol*np.sqrt(self.dt)*returns1)
            S2 = S1[i] * R2

            # Compute portfolio payoff as weighted sum of calls
            payoff_total = np.zeros(self.num_paths)
            delta_pathwise_total = np.zeros(self.num_paths)

            for strike, weight in zip(self.strikes, self.weights):
                payoff = np.maximum(S2 - strike, 0.0)
                indicator = (S2 > strike).astype(np.float64)
                payoff_total += weight * payoff
                # Pathwise delta for Black-Scholes: indicator * R2
                delta_pathwise_total += weight * indicator * R2

            # Price: average payoff
            y_prices[i, 0] = payoff_total.mean()

            # Pathwise delta: average of (indicator * R2) weighted
            dydx_pathwise[i, 0] = delta_pathwise_total.mean()

            # LRM Delta: E[payoff * score] where score = Z / (vol * sqrt(dt))
            score = returns1 / (self.vol * np.sqrt(self.dt))
            dydx_lrm[i, 0] = (payoff_total * score).mean()

            # PW-LR Gamma: indicator * (S2/S1^2) * (Z/(sigma*sqrt(T)) - 1)
            # This applies LRM to the pathwise indicator, not to the payoff!
            # Based on collective_experiments.py line 2339
            gamma_pwlr_total = np.zeros(self.num_paths)
            for strike, weight in zip(self.strikes, self.weights):
                indicator = (S2 > strike).astype(np.float64)
                gamma_contribution = indicator * (S2 / (S1[i]**2)) * (score - 1.0)
                gamma_pwlr_total += weight * gamma_contribution
            d2ydx2_lrm[i, 0] = gamma_pwlr_total.mean()

        return S1.reshape(-1, 1), y_prices, dydx_pathwise, dydx_lrm, d2ydx2_lrm

    def testSet(self, num=PORTFOLIO_NUM_TEST, lower=0.5, upper=1.5, seed=None):
        """Analytical test set using Black-Scholes formulas"""
        rng = np.random.RandomState(seed)
        spots = rng.uniform(low=lower, high=upper, size=num)

        prices = np.zeros(num)
        deltas = np.zeros(num)
        gammas = np.zeros(num)

        for i, spot in enumerate(spots):
            for strike, weight in zip(self.strikes, self.weights):
                prices[i] += weight * bsPrice(spot, strike, self.vol, self.dt)
                deltas[i] += weight * bsDelta(spot, strike, self.vol, self.dt)
                gammas[i] += weight * bsGamma(spot, strike, self.vol, self.dt)

        return spots.reshape(-1, 1), prices.reshape(-1, 1), deltas.reshape(-1, 1), gammas.reshape(-1, 1)


class DigitalPortfolioOption:
    """Digital portfolio: D(0.75) - D(0.95) + D(1.15) - D(1.35)

    Uses Black-Scholes (geometric Brownian motion) to be consistent with
    other single-asset experiments in this notebook. We treat this portfolio
    as an LRM-only example since pathwise estimators (and therefore PW-LR)
    are biased for digital payoffs.
    """

    def __init__(self, vol=PORTFOLIO_VOL, T1=PORTFOLIO_T1, T2=PORTFOLIO_T2, strikes=None, weights=None,
                 volMult=PORTFOLIO_VOL_MULT, num_paths=PORTFOLIO_NUM_PATHS_PER_S1, seed=None):
        self.spot = PORTFOLIO_SPOT
        self.vol = vol
        self.T1 = T1
        self.T2 = T2
        self.dt = T2 - T1
        self.strikes = strikes if strikes is not None else PORTFOLIO_DIGITAL_STRIKES
        self.weights = weights if weights is not None else PORTFOLIO_DIGITAL_WEIGHTS
        self.volMult = volMult
        self.num_paths = num_paths
        self.rng = np.random.RandomState(seed)

    def trainingSet(self, m, seed=None):
        """Generate training data with LRM delta and gamma (no pathwise for digital)
        Uses Black-Scholes (geometric Brownian motion)
        """
        rng = np.random.RandomState(seed)

        # Sample S1 with volMult for better coverage (geometric Brownian motion)
        returns0 = rng.normal(size=m)
        vol0 = self.vol * self.volMult
        R1 = np.exp(-0.5*vol0*vol0*self.T1 + vol0*np.sqrt(self.T1)*returns0)
        S1 = self.spot * R1

        # For each S1, average over num_paths draws for S2
        y_prices = np.zeros((m, 1))
        dydx_pathwise = np.zeros((m, 1))  # Zero for digitals (discontinuous)
        dydx_lrm = np.zeros((m, 1))
        d2ydx2_lrm = np.zeros((m, 1))

        for i in range(m):
            returns1 = rng.normal(size=self.num_paths)
            # Geometric Brownian motion: S2 = S1 * exp(-0.5*vol^2*dt + vol*sqrt(dt)*Z)
            R2 = np.exp(-0.5*self.vol*self.vol*self.dt + self.vol*np.sqrt(self.dt)*returns1)
            S2 = S1[i] * R2

            # Compute portfolio payoff as weighted sum of digitals
            payoff_total = np.zeros(self.num_paths)

            for strike, weight in zip(self.strikes, self.weights):
                indicator = (S2 > strike).astype(np.float64)
                payoff_total += weight * indicator

            # Price: average payoff
            y_prices[i, 0] = payoff_total.mean()

            # Pathwise delta: zero for digitals (discontinuous payoff)
            dydx_pathwise[i, 0] = 0.0

            # LRM Delta: E[indicator * score] where score = Z / (S1 * vol * sqrt(dt))
            # For Black-Scholes digital options, the score includes S1 in denominator
            # Based on lrm_derivative function (line 991-992) and digitalTrainingSet (line 1119)
            score_lrm = returns1 / (S1[i] * self.vol * np.sqrt(self.dt))
            dydx_lrm[i, 0] = (payoff_total * score_lrm).mean()

            # PW-LR Gamma for digitals: indicator * (S2/S1^2) * (Z/(sigma*sqrt(T)) - 1)
            # For gamma, we use the same score as for calls: Z / (vol * sqrt(dt))
            # Based on collective_experiments.py line 2339
            score_gamma = returns1 / (self.vol * np.sqrt(self.dt))
            gamma_pwlr_total = np.zeros(self.num_paths)
            for strike, weight in zip(self.strikes, self.weights):
                indicator = (S2 > strike).astype(np.float64)
                gamma_contribution = indicator * (S2 / (S1[i]**2)) * (score_gamma - 1.0)
                gamma_pwlr_total += weight * gamma_contribution
            d2ydx2_lrm[i, 0] = gamma_pwlr_total.mean()

        return S1.reshape(-1, 1), y_prices, dydx_pathwise, dydx_lrm, d2ydx2_lrm

    def testSet(self, num=PORTFOLIO_NUM_TEST, lower=0.5, upper=1.5, seed=None):
        """Analytical test set using Black-Scholes formulas"""
        rng = np.random.RandomState(seed)
        spots = rng.uniform(low=lower, high=upper, size=num)

        prices = np.zeros(num)
        deltas = np.zeros(num)
        gammas = np.zeros(num)

        for i, spot in enumerate(spots):
            for strike, weight in zip(self.strikes, self.weights):
                prices[i] += weight * digitalPrice(spot, strike, self.vol, self.dt)
                deltas[i] += weight * digitalDelta(spot, strike, self.vol, self.dt)
                gammas[i] += weight * digitalGamma(spot, strike, self.vol, self.dt)

        return spots.reshape(-1, 1), prices.reshape(-1, 1), deltas.reshape(-1, 1), gammas.reshape(-1, 1)

def test_portfolio(generator, sizes, nTest, simulSeed=None, testSeed=None, weightSeed=None):
    """
    Test portfolio options with multiple methods:
    1. Standard ML
    2. Delta-Pathwise (for calls only)
    3. Delta-LRM
    4. PW-LR ML with gamma
    """

    # Generate training data
    xTrain, yTrain, dydxTrain_pathwise, dydxTrain_lrm, d2ydx2Train = \
        generator.trainingSet(max(sizes), seed=simulSeed)

    xTest, yTest, dydxTest, d2ydx2Test = generator.testSet(num=nTest, seed=testSeed)

    # Determine portfolio type
    is_call = isinstance(generator, CallPortfolioOption)

    # Neural approximators - create fresh instances for each size
    predvalues = {}
    preddeltas = {}
    predgammas = {}

    for size in sizes:

        # Create fresh regressors for each size to avoid state issues
        regressor_standard = Neural_Approximator(xTrain, yTrain, None, None)

        regressor_lrm = Neural_Approximator(xTrain, yTrain, dydxTrain_lrm, None)

        regressor_pathwise = None
        regressor_pwlr = None
        if is_call:
            regressor_pathwise = Neural_Approximator(xTrain, yTrain, dydxTrain_pathwise, None)
            regressor_pwlr = Neural_Approximator(xTrain, yTrain, dydxTrain_pathwise, d2ydx2Train)

        # 1. Standard ML
        regressor_standard.prepare(size, False, weight_seed=weightSeed)
        regressor_standard.train("standard training")
        predictions, deltas, gammas = regressor_standard.predict_values_derivs_and_gamma(xTest)
        predvalues[("standard", size)] = predictions
        preddeltas[("standard", size)] = deltas[:, 0]
        predgammas[("standard", size)] = gammas[:, 0]

        # 2. Delta-Pathwise (for calls only)
        if is_call:
            regressor_pathwise.prepare(size, True, use_gamma=False, weight_seed=weightSeed)
            regressor_pathwise.train("pathwise training")
            predictions, deltas, gammas = regressor_pathwise.predict_values_derivs_and_gamma(xTest)
            predvalues[("pathwise", size)] = predictions
            preddeltas[("pathwise", size)] = deltas[:, 0]
            predgammas[("pathwise", size)] = gammas[:, 0]

        # 3. Delta-LRM
        regressor_lrm.prepare(size, True, use_gamma=False, weight_seed=weightSeed)
        regressor_lrm.train("lrm training")
        predictions, deltas, gammas = regressor_lrm.predict_values_derivs_and_gamma(xTest)
        predvalues[("lrm", size)] = predictions
        preddeltas[("lrm", size)] = deltas[:, 0]
        predgammas[("lrm", size)] = gammas[:, 0]

        # 4. PW-LR ML with gamma (calls only)
        if is_call and regressor_pwlr is not None:
            regressor_pwlr.prepare(size, True, use_gamma=True, weight_seed=weightSeed)
            regressor_pwlr.train("pw-lr training")
            predictions, deltas, gammas = regressor_pwlr.predict_values_derivs_and_gamma(xTest)
            predvalues[("pwlr", size)] = predictions
            preddeltas[("pwlr", size)] = deltas[:, 0]
            predgammas[("pwlr", size)] = gammas[:, 0]

    return xTest, yTest, dydxTest, d2ydx2Test, predvalues, preddeltas, predgammas

def run_portfolio_experiment_with_seeds(generator, sizes, nTest, num_seeds=100, weightSeed=None):
    """
    Run portfolio options experiment multiple times with different seeds

    Returns averaged predictions and statistics across all seeds
    """
    # Determine portfolio type
    is_call = isinstance(generator, CallPortfolioOption)
    methods = ["standard", "pathwise", "lrm", "pwlr"] if is_call else ["standard", "lrm"]

    # Storage for predictions and RMSE values across seeds
    all_predictions_values = {method: {size: [] for size in sizes} for method in methods}
    all_predictions_deltas = {method: {size: [] for size in sizes} for method in methods}
    all_predictions_gammas = {method: {size: [] for size in sizes} for method in methods}
    rmse_values = {method: {size: [] for size in sizes} for method in methods}
    rmse_deltas = {method: {size: [] for size in sizes} for method in methods}
    rmse_gammas = {method: {size: [] for size in sizes} for method in methods}

    # Run experiment for each seed with progress bar
    try:
        progress_iter = tqdm(range(num_seeds), desc="running portfolio seeds")
        use_tqdm = True
    except NameError:
        # Fallback if tqdm not available
        progress_iter = range(num_seeds)
        use_tqdm = False
        print(f"Running portfolio experiment with {num_seeds} seeds...")

    for seed_idx in progress_iter:
        if not use_tqdm and (seed_idx + 1) % 10 == 0:
            print(f"  Completed {seed_idx + 1}/{num_seeds} seeds")

        simulSeed = np.random.randint(0, 10000)

        # Run experiment - use fixed testSeed=42 so test set is same across all seeds
        xTest, yTest, dydxTest, d2ydx2Test, predvalues, preddeltas, predgammas = \
            test_portfolio(generator, sizes, nTest, simulSeed, 42, weightSeed)

        # Store predictions and compute RMSE for each method and size
        for method in methods:
            for size in sizes:
                # Store predictions
                all_predictions_values[method][size].append(predvalues[(method, size)])
                all_predictions_deltas[method][size].append(preddeltas[(method, size)])
                all_predictions_gammas[method][size].append(predgammas[(method, size)])

                # Price RMSE
                rmse_val = np.sqrt(((predvalues[(method, size)] - yTest)**2).mean())
                rmse_values[method][size].append(rmse_val)

                # Delta RMSE
                rmse_delt = np.sqrt(((preddeltas[(method, size)] - dydxTest[:, 0])**2).mean())
                rmse_deltas[method][size].append(rmse_delt)

                # Gamma RMSE
                rmse_gam = np.sqrt(((predgammas[(method, size)] - d2ydx2Test[:, 0])**2).mean())
                rmse_gammas[method][size].append(rmse_gam)

    # Compute averaged predictions
    avg_values = {}
    avg_deltas = {}
    avg_gammas = {}
    for method in methods:
        for size in sizes:
            avg_values[(method, size)] = np.mean(all_predictions_values[method][size], axis=0)
            avg_deltas[(method, size)] = np.mean(all_predictions_deltas[method][size], axis=0)
            avg_gammas[(method, size)] = np.mean(all_predictions_gammas[method][size], axis=0)

    # Compute statistics
    stats = {}
    for metric, data in [("price", rmse_values), ("delta", rmse_deltas), ("gamma", rmse_gammas)]:
        stats[metric] = {}
        for method in methods:
            stats[metric][method] = {}
            for size in sizes:
                values_array = np.array(data[method][size])
                stats[metric][method][size] = {
                    'mean': values_array.mean(),
                    'std': values_array.std(),
                    'min': values_array.min(),
                    'max': values_array.max()
                }

    return xTest, yTest, dydxTest, d2ydx2Test, stats, avg_values, avg_deltas, avg_gammas

def print_portfolio_statistics(stats, sizes, is_call=True):
    """Print formatted statistics for multiple seed portfolio runs"""
    methods = ["standard", "pathwise", "lrm", "pwlr"] if is_call else ["standard", "lrm"]

    for metric in ["price", "delta", "gamma"]:
        print(f"\n{metric.upper()} RMSE (over {NUM_SEEDS} seeds):")
        print(f"{'Method':<15} {'Size':<10} {'Mean':<12} {'Std':<12} {'Min':<12} {'Max':<12}")
        print("-"*70)

        for method in methods:
            for size in sizes:
                s = stats[metric][method][size]
                print(f"{method:<15} {size:<10} {s['mean']:<12.6f} {s['std']:<12.6f} {s['min']:<12.6f} {s['max']:<12.6f}")

def graph_portfolio(title, predictions, xAxis, xAxisName, yAxisName,
                    targets, sizes, computeRmse=False, is_call=True):
    """
    Graph predictions for portfolio options
    """

    numRows = len(sizes)
    numCols = 4 if is_call else 2  # Calls: standard, pathwise, lrm, pwlr; Digitals: standard, lrm

    fig, ax = plt.subplots(numRows, numCols, squeeze=False)
    fig.set_size_inches(4 * numCols + 1.5, 4 * numRows)

    for i, size in enumerate(sizes):
        ax[i,0].annotate("size %d" % size, xy=(0, 0.5),
          xytext=(-ax[i,0].yaxis.labelpad-5, 0),
          xycoords=ax[i,0].yaxis.label, textcoords='offset points',
          ha='right', va='center')

    if is_call:
        ax[0,0].set_title("standard")
        ax[0,1].set_title("pathwise")
        ax[0,2].set_title("lrm")
        ax[0,3].set_title("pw-lr")
        methods = ["standard", "pathwise", "lrm", "pwlr"]
    else:
        ax[0,0].set_title("standard")
        ax[0,1].set_title("lrm")
        methods = ["standard", "lrm"]

    for i, size in enumerate(sizes):
        for j, regType in enumerate(methods):

            if computeRmse:
                errors = 100 * (predictions[(regType, size)] - targets)
                rmse = np.sqrt((errors ** 2).mean(axis=0))
                # Extract scalar value if rmse is an array
                if isinstance(rmse, np.ndarray):
                    rmse = rmse.item() if rmse.size == 1 else rmse[0]
                t = "rmse %.2f" % rmse
            else:
                t = xAxisName

            ax[i,j].set_xlabel(t)
            ax[i,j].set_ylabel(yAxisName)

            # Ensure predictions and targets are 1D arrays for plotting
            pred_plot = predictions[(regType, size)]
            if pred_plot.ndim > 1:
                pred_plot = pred_plot.flatten()
            targets_plot = targets
            if targets_plot.ndim > 1:
                targets_plot = targets_plot.flatten()

            # Sort by x-axis for cleaner plots (like portfolio_options.py)
            xAxis_flat = xAxis.flatten() if xAxis.ndim > 1 else xAxis
            sort_idx = np.argsort(xAxis_flat)
            xAxis_sorted = xAxis_flat[sort_idx]
            pred_plot_sorted = pred_plot[sort_idx]
            targets_plot_sorted = targets_plot[sort_idx]

            ax[i,j].plot(xAxis_sorted*100, pred_plot_sorted*100, 'co',
                        markersize=2, markerfacecolor='white', markeredgewidth=0.5)
            ax[i,j].plot(xAxis_sorted*100, targets_plot_sorted*100, 'r-', linewidth=1)

    plt.tight_layout()
    plt.subplots_adjust(top=0.9)
    plt.suptitle("%s -- %s" % (title, yAxisName), fontsize=16)
    plt.show()

## Portfolio Options Experiments

We test both call and digital portfolios to show that differential ML methods
(especially PW-LR with gamma) outperform standard ML on complex payoff structures.

In [ ]:
%matplotlib inline

# Configuration for portfolio experiments
sizes = PORTFOLIO_SAMPLE_SIZES
nTest = PORTFOLIO_NUM_TEST
showDeltas = True
showGammas = True
weightSeed = WEIGHT_SEED

print("\n" + "="*80)
print("PORTFOLIO OPTIONS EXPERIMENTS")
print("="*80)
print("\nTesting Call Portfolio: C(0.85) - 1.5*C(0.9) + 0.75*C(1.15)")
print("Testing Digital Portfolio: D(0.75) - D(0.95) + D(1.15) - D(1.35)")

# Call Portfolio Experiment
print("\n" + "="*80)
print("CALL PORTFOLIO EXPERIMENT")
print("="*80 + "\n")

generator_call = CallPortfolioOption(vol=PORTFOLIO_VOL, T1=PORTFOLIO_T1, T2=PORTFOLIO_T2,
                                     strikes=PORTFOLIO_CALL_STRIKES, weights=PORTFOLIO_CALL_WEIGHTS,
                                     volMult=PORTFOLIO_VOL_MULT, num_paths=PORTFOLIO_NUM_PATHS_PER_S1, seed=None)

if RUN_MULTIPLE_SEEDS:
    # Run with multiple seeds, show statistics and averaged graphs
    xAxis_call, yTest_call, dydxTest_call, d2ydx2Test_call, stats_call, avg_values_call, avg_deltas_call, avg_gammas_call = \
        run_portfolio_experiment_with_seeds(generator_call, sizes, nTest, NUM_SEEDS, weightSeed)

    # Show statistics
    print_portfolio_statistics(stats_call, sizes, is_call=True)

    # Show averaged predictions
    graph_portfolio("Call Portfolio (averaged over %d seeds)" % NUM_SEEDS, avg_values_call, xAxis_call, "", "values", yTest_call, sizes, True, is_call=True)

    if showDeltas:
        graph_portfolio("Call Portfolio Deltas (averaged over %d seeds)" % NUM_SEEDS, avg_deltas_call, xAxis_call, "", "deltas", dydxTest_call[:, 0:1], sizes, True, is_call=True)

    if showGammas:
        graph_portfolio("Call Portfolio Gammas (averaged over %d seeds)" % NUM_SEEDS, avg_gammas_call, xAxis_call, "", "gammas", d2ydx2Test_call[:, 0:1], sizes, True, is_call=True)
else:
    # Single run with graphs
    simulSeed = np.random.randint(0, 10000)
    print("using seed %d" % simulSeed)

    xAxis_call, yTest_call, dydxTest_call, d2ydx2Test_call, values_call, deltas_call, gammas_call = \
        test_portfolio(generator_call, sizes, nTest, simulSeed, None, weightSeed)

    # Show predictions - prices
    graph_portfolio("Call Portfolio", values_call, xAxis_call, "", "values", yTest_call, sizes, True, is_call=True)

    # Show predictions - deltas
    if showDeltas:
        graph_portfolio("Call Portfolio", deltas_call, xAxis_call, "", "deltas", dydxTest_call[:, 0:1], sizes, True, is_call=True)

    # Show predictions - gammas
    if showGammas:
        graph_portfolio("Call Portfolio", gammas_call, xAxis_call, "", "gammas", d2ydx2Test_call[:, 0:1], sizes, True, is_call=True)

# Digital Portfolio Experiment
print("\n" + "="*80)
print("DIGITAL PORTFOLIO EXPERIMENT")
print("="*80 + "\n")

generator_digital = DigitalPortfolioOption(vol=PORTFOLIO_VOL, T1=PORTFOLIO_T1, T2=PORTFOLIO_T2,
                                           strikes=PORTFOLIO_DIGITAL_STRIKES, weights=PORTFOLIO_DIGITAL_WEIGHTS,
                                           volMult=PORTFOLIO_VOL_MULT, num_paths=PORTFOLIO_NUM_PATHS_PER_S1, seed=None)

if RUN_MULTIPLE_SEEDS:
    # Run with multiple seeds, show statistics and averaged graphs
    xAxis_digital, yTest_digital, dydxTest_digital, d2ydx2Test_digital, stats_digital, avg_values_digital, avg_deltas_digital, avg_gammas_digital = \
        run_portfolio_experiment_with_seeds(generator_digital, sizes, nTest, NUM_SEEDS, weightSeed)

    # Show statistics
    print_portfolio_statistics(stats_digital, sizes, is_call=False)

    # Show averaged predictions
    graph_portfolio("Digital Portfolio (averaged over %d seeds)" % NUM_SEEDS, avg_values_digital, xAxis_digital, "", "values", yTest_digital, sizes, True, is_call=False)

    if showDeltas:
        graph_portfolio("Digital Portfolio Deltas (averaged over %d seeds)" % NUM_SEEDS, avg_deltas_digital, xAxis_digital, "", "deltas", dydxTest_digital[:, 0:1], sizes, True, is_call=False)

    if showGammas:
        graph_portfolio("Digital Portfolio Gammas (averaged over %d seeds)" % NUM_SEEDS, avg_gammas_digital, xAxis_digital, "", "gammas", d2ydx2Test_digital[:, 0:1], sizes, True, is_call=False)
else:
    # Single run with graphs
    simulSeed = np.random.randint(0, 10000)
    print("using seed %d" % simulSeed)

    xAxis_digital, yTest_digital, dydxTest_digital, d2ydx2Test_digital, values_digital, deltas_digital, gammas_digital = \
        test_portfolio(generator_digital, sizes, nTest, simulSeed, None, weightSeed)

    # Show predictions - prices
    graph_portfolio("Digital Portfolio", values_digital, xAxis_digital, "", "values", yTest_digital, sizes, True, is_call=False)

    # Show predictions - deltas
    if showDeltas:
        graph_portfolio("Digital Portfolio", deltas_digital, xAxis_digital, "", "deltas", dydxTest_digital[:, 0:1], sizes, True, is_call=False)

    # Show predictions - gammas
    if showGammas:
        graph_portfolio("Digital Portfolio", gammas_digital, xAxis_digital, "", "gammas", d2ydx2Test_digital[:, 0:1], sizes, True, is_call=False)
